# Análise exploratória | ZVCMM103 | tbl_ds_pro_zvcmm103_parada_manutencao

| Propriedade | Valor |
|---|---|
| Tabela | ``dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`` |
| Ambiente | DEV |
| Objeto | Tabela |
| Colunas do DESCRIBE | 196 |
| Tipos | boolean, date, decimal(13,3), decimal(17,4), decimal(17,5), double, int, string |
| Clustering declarado | Não informado no CSV |
| Centro | `cod_centro` |
| Chave candidata | `num_documento` + `tp_tipo_documento` + `num_sequencial_alteracoes` (validar na seção 4) |

**Como usar:** importar no Databricks e executar Run All. Células SQL independentes, somente leitura. Resultados são medições, não conclusões prévias. Para views, conferir no DESCRIBE EXTENDED se filtros da tela SAP são aplicados. Não executar cenários antes de analisar os resultados. Fontes: `DESCRIBE_ZVCMM103.csv` e `01_PADROES_SQL_NOTEBOOKS.md`.

### Alertas do schema
| Severidade | Alerta |
|---|---|
| 🔴 ALTA | 25 campos tratados como potencialmente pessoais; amostras só com hash ou omitidos. |
| 🟡 MÉDIA | Valores DOUBLE: conferir precisão e tolerância de 0,005 na comparação. |

### Índice
1 Metadados, 2 Volumetria, 3 Distribuição por centro, 4 Granularidade, 5 Duplicidade, 6 Preenchimento de todas as colunas, 6.1 Colunas sem fonte, 7 Cardinalidade, 8 Domínio, 9 Booleanas e indicadores, 10 Perfil numérico, 11 Totais para conciliação, 12 Datas, 13 Códigos, 14 Checksum, 15 Amostra, 16 Distribuição interna, 17 Freshness, 18 Específicas da transação, 19 Amostra ampliada, 20 Resumo. Inclui subseções 3.1 e 6.1.


## 1. Metadados

Inspeciona propriedades e comentário do objeto para conferir escopo e eventuais filtros da transação.


In [0]:
DESCRIBE EXTENDED `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`;


### 1.1. Detalhes e histórico da tabela

Comandos independentes, executados somente em tabelas.


In [0]:
DESCRIBE DETAIL `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`;


In [0]:
DESCRIBE HISTORY `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`;


## 2. Volumetria

Conta registros e entidades existentes, sem supor que campos de identificação formam uma chave.


In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_fornecedor`) AS distintos_cod_fornecedor, COUNT(DISTINCT `num_documento`) AS distintos_num_documento, COUNT(DISTINCT `cod_requisicao_compra`) AS distintos_cod_requisicao_compra, COUNT(DISTINCT `num_material`) AS distintos_num_material, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`;


## 3. Distribuição por centro

Mede volumes e sinaliza centros entre 10 mil e 300 mil registros, sem criar cenários.


In [0]:
SELECT `cod_centro` AS centro, COUNT(*) AS linhas, CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE 'AVALIAR VOLUME' END AS avaliacao FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_centro` ORDER BY linhas DESC;


## 3.1. Centros 4128 e 4014

Confirma a presença e o formato literal dos centros padrão; ainda não filtra os demais testes.


In [0]:
SELECT `cod_centro` AS centro_literal, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) AS tamanho, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` WHERE TRIM(CAST(`cod_centro` AS STRING)) RLIKE '^0*(4128|4014)$' GROUP BY `cod_centro`, LENGTH(TRIM(CAST(`cod_centro` AS STRING))) ORDER BY centro_literal;


## 4. Granularidade

Compara 3 a 5 chaves candidatas. Uma razão igual a 1 é indício, não prova isolada de chave de negócio.


In [0]:
WITH d AS (SELECT 'num_documento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) k UNION ALL SELECT 'num_documento + tp_tipo_documento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `tp_tipo_documento` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) k UNION ALL SELECT 'num_documento + tp_tipo_documento + num_sequencial_alteracoes' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) k UNION ALL SELECT 'num_documento + tp_tipo_documento + num_sequencial_alteracoes + cod_evento' AS chave_candidata, COUNT(*) AS chaves_distintas FROM (SELECT DISTINCT `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`, `cod_evento` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) k), v AS (SELECT COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) SELECT d.chave_candidata, v.linhas, d.chaves_distintas, ROUND(v.linhas / NULLIF(d.chaves_distintas, 0), 4) AS linhas_por_chave FROM d CROSS JOIN v ORDER BY linhas_por_chave, chave_candidata;


## 5. Duplicidade

Separa repetições de chave entre linhas idênticas e versões diferentes, com hash da linha inteira sem exibir dados pessoais.


In [0]:
WITH h AS (SELECT t.`num_documento`, t.`tp_tipo_documento`, t.`num_sequencial_alteracoes`, md5(to_json(struct(t.`cod_gl_mm_data_da_aprovacao`, t.`num_documento`, t.`tp_tipo_documento`, t.`num_sequencial_alteracoes`, t.`cod_evento`, t.`cod_empresa`, t.`nm_empresa`, t.`cod_exercicio`, t.`cod_unidade_negocios`, t.`num_documento_inventario`, t.`cod_requisicao_compra`, t.`cod_item_requisicao_compra`, t.`num_acompanhamento`, t.`cod_classificacao_contabil_req`, t.`cod_classe_custo_req`, t.`num_ordem_req`, t.`desc_texto_breve_req`, t.`num_pedido_compras`, t.`num_item_pedido_compras`, t.`tp_tipo_pedido`, t.`cod_classificacao_contabil`, t.`cod_classe_custo`, t.`cod_classificacao_contabil_ped`, t.`cod_classe_custo_ped`, t.`num_ordem_ped`, t.`desc_texto_breve_ped`, t.`num_ordem`, t.`num_ordem_interna`, t.`cod_categoria_item`, t.`cod_elemento_pep`, t.`num_contrato`, t.`num_item_contrato`, t.`num_contrato_re`, t.`num_item_contrato_re`, t.`cod_fornecedor`, t.`cod_fornecedor_re`, t.`nm_fornecedor_re`, t.`num_material`, t.`desc_texto_breve`, t.`num_folha_servicos`, t.`num_servico`, t.`num_servico_re`, t.`sg_unidade_medida`, t.`sg_unidade_medida_base`, t.`cod_centro`, t.`cod_centro_ped`, t.`cod_centro_custo`, t.`desc_denominacao`, t.`desc_texto_breve_re`, t.`num_local_instalacao`, t.`num_equipamento`, t.`cod_org_compras`, t.`cod_grupo_compradores`, t.`vl_valor_bruto`, t.`vl_limite_alcada`, t.`qt_quantidade`, t.`vl_fixado_cabecalho`, t.`vl_preco_liquido_ped`, t.`vl_preco_bruto`, t.`sg_moeda`, t.`sg_moeda_ped`, t.`cod_estrategia_liberacao`, t.`cod_status_liberacao`, t.`cod_grupo_liberacao`, t.`cod_grupo_liberador`, t.`cod_liberacao`, t.`cod_nivel_alcada`, t.`cod_meio_aprovacao`, t.`tp_movimento`, t.`desc_cod_liberacao_01`, t.`desc_cod_liberacao_02`, t.`desc_cod_liberacao_03`, t.`desc_cod_liberacao_04`, t.`desc_cod_liberacao_05`, t.`desc_cod_liberacao_06`, t.`desc_cod_liberacao_07`, t.`desc_cod_liberacao_08`, t.`cod_cadeia_aprovacao_n1`, t.`cod_cadeia_aprovacao_n2`, t.`cod_cadeia_aprovacao_n3`, t.`cod_cadeia_aprovacao_n4`, t.`cod_cadeia_aprovacao_n5`, t.`cod_cadeia_aprovacao_n6`, t.`cod_cadeia_aprovacao_n7`, t.`cod_cadeia_aprovacao_n8`, t.`cod_nivel_n1`, t.`cod_nivel_n2`, t.`cod_nivel_n3`, t.`cod_nivel_n4`, t.`cod_nivel_n5`, t.`cod_nivel_n6`, t.`cod_nivel_n7`, t.`cod_nivel_n8`, t.`nm_aprovador_01`, t.`nm_aprovador_02`, t.`nm_aprovador_03`, t.`nm_aprovador_04`, t.`nm_aprovador_05`, t.`nm_aprovador_06`, t.`nm_aprovador_07`, t.`nm_aprovador_08`, t.`cod_liberacao_01`, t.`cod_liberacao_02`, t.`cod_liberacao_03`, t.`cod_liberacao_04`, t.`cod_liberacao_05`, t.`cod_liberacao_06`, t.`cod_liberacao_07`, t.`cod_liberacao_08`, t.`cod_usuario_lib_01`, t.`cod_usuario_lib_02`, t.`cod_usuario_lib_03`, t.`cod_usuario_lib_04`, t.`cod_usuario_lib_05`, t.`cod_usuario_lib_06`, t.`cod_usuario_lib_07`, t.`cod_usuario_lib_08`, t.`dt_recebimento_01`, t.`dt_recebimento_02`, t.`dt_recebimento_03`, t.`dt_recebimento_04`, t.`dt_recebimento_05`, t.`dt_recebimento_06`, t.`dt_recebimento_07`, t.`dt_recebimento_08`, t.`hr_recebimento_01`, t.`hr_recebimento_02`, t.`hr_recebimento_03`, t.`hr_recebimento_04`, t.`hr_recebimento_05`, t.`hr_recebimento_06`, t.`hr_recebimento_07`, t.`hr_recebimento_08`, t.`dt_recebimento_msg`, t.`hr_recebimento_msg`, t.`dt_liberacao_01`, t.`dt_liberacao_02`, t.`dt_liberacao_03`, t.`dt_liberacao_04`, t.`dt_liberacao_05`, t.`dt_liberacao_06`, t.`dt_liberacao_07`, t.`dt_liberacao_08`, t.`dt_liberacao_final`, t.`hr_liberacao_01`, t.`hr_liberacao_02`, t.`hr_liberacao_03`, t.`hr_liberacao_04`, t.`hr_liberacao_05`, t.`hr_liberacao_06`, t.`hr_liberacao_07`, t.`hr_liberacao_08`, t.`hr_liberacao_final`, t.`hr_entrada`, t.`dt_lancamento`, t.`dt_remessa`, t.`nm_requisitante`, t.`nm_solicitante`, t.`nm_usuario_aprovador`, t.`cod_criado_por`, t.`cod_usuario_substituido`, t.`cod_proximo_aprovador`, t.`cod_usuario_grupo_aprovador`, t.`ind_delegacao_ativa`, t.`cod_responsavel_delegacao`, t.`dt_validade_delegacao`, t.`cod_status_atual`, t.`cod_status_portal`, t.`cod_tempo_decorrido_01`, t.`cod_tempo_decorrido_02`, t.`cod_tempo_decorrido_03`, t.`cod_tempo_decorrido_04`, t.`cod_tempo_decorrido_05`, t.`cod_tempo_decorrido_06`, t.`cod_tempo_decorrido_07`, t.`cod_tempo_decorrido_08`, t.`qt_dias_expirar`, t.`qt_dias_faltantes_aprovacao`, t.`qt_dias_total_aprovacao`, t.`ind_eliminacao`, t.`ind_liberacao`, t.`ind_liberacao_incompleta`, t.`ind_bloqueio_riscos`, t.`desc_bloqueio_riscos`, t.`vl_indice_definitivo_novo`, t.`vl_indice_definitivo_antigo`, t.`dt_indice_inflacao`, t.`pct_indice_inflacao_novo`, t.`pct_indice_inflacao_antigo`, t.`cod_indice_inflacao`, t.`cod_versao_indice_inflacao`, t.`system_date`, t.`timestamp`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` t), d AS (SELECT `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`, COUNT(*) AS linhas, COUNT(DISTINCT hash_linha) AS versoes FROM h GROUP BY `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COALESCE(SUM(linhas),0) AS linhas_em_chaves_repetidas, COALESCE(SUM(linhas - versoes),0) AS repeticoes_identicas, COALESCE(SUM(CASE WHEN versoes > 1 THEN 1 ELSE 0 END),0) AS chaves_com_versoes_diferentes FROM d;


## 5.1. Atributos variáveis nas chaves repetidas

Mede variação de atributos não pessoais nas chaves candidatas repetidas; ausência de variação não prova identidade da linha completa.


In [0]:
WITH d AS (SELECT `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`, COUNT(*) AS linhas, COUNT(DISTINCT CAST(`cod_gl_mm_data_da_aprovacao` AS STRING)) AS `dist_cod_gl_mm_data_da_aprovacao`, COUNT(DISTINCT CAST(`num_documento` AS STRING)) AS `dist_num_documento`, COUNT(DISTINCT CAST(`tp_tipo_documento` AS STRING)) AS `dist_tp_tipo_documento`, COUNT(DISTINCT CAST(`num_sequencial_alteracoes` AS STRING)) AS `dist_num_sequencial_alteracoes`, COUNT(DISTINCT CAST(`cod_evento` AS STRING)) AS `dist_cod_evento`, COUNT(DISTINCT CAST(`cod_empresa` AS STRING)) AS `dist_cod_empresa`, COUNT(DISTINCT CAST(`nm_empresa` AS STRING)) AS `dist_nm_empresa`, COUNT(DISTINCT CAST(`cod_exercicio` AS STRING)) AS `dist_cod_exercicio`, COUNT(DISTINCT CAST(`cod_unidade_negocios` AS STRING)) AS `dist_cod_unidade_negocios`, COUNT(DISTINCT CAST(`num_documento_inventario` AS STRING)) AS `dist_num_documento_inventario`, COUNT(DISTINCT CAST(`cod_requisicao_compra` AS STRING)) AS `dist_cod_requisicao_compra`, COUNT(DISTINCT CAST(`cod_item_requisicao_compra` AS STRING)) AS `dist_cod_item_requisicao_compra` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes` HAVING COUNT(*) > 1) SELECT COUNT(*) AS chaves_repetidas, COUNT_IF(`dist_cod_gl_mm_data_da_aprovacao` > 1) AS `chaves_variam_cod_gl_mm_data_da_aprovacao`, COUNT_IF(`dist_num_documento` > 1) AS `chaves_variam_num_documento`, COUNT_IF(`dist_tp_tipo_documento` > 1) AS `chaves_variam_tp_tipo_documento`, COUNT_IF(`dist_num_sequencial_alteracoes` > 1) AS `chaves_variam_num_sequencial_alteracoes`, COUNT_IF(`dist_cod_evento` > 1) AS `chaves_variam_cod_evento`, COUNT_IF(`dist_cod_empresa` > 1) AS `chaves_variam_cod_empresa`, COUNT_IF(`dist_nm_empresa` > 1) AS `chaves_variam_nm_empresa`, COUNT_IF(`dist_cod_exercicio` > 1) AS `chaves_variam_cod_exercicio`, COUNT_IF(`dist_cod_unidade_negocios` > 1) AS `chaves_variam_cod_unidade_negocios`, COUNT_IF(`dist_num_documento_inventario` > 1) AS `chaves_variam_num_documento_inventario`, COUNT_IF(`dist_cod_requisicao_compra` > 1) AS `chaves_variam_cod_requisicao_compra`, COUNT_IF(`dist_cod_item_requisicao_compra` > 1) AS `chaves_variam_cod_item_requisicao_compra` FROM d;


## 6. Preenchimento de todas as colunas

Mede nulos, vazios textuais, zeros e úteis em TODAS as colunas. Zero em código pode ser valor legítimo; veredito não equivale a defeito.


In [0]:
WITH perf AS (SELECT stack(24, 'cod_gl_mm_data_da_aprovacao', 'string', c0_n, c0_v, c0_z, 'num_documento', 'string', c1_n, c1_v, c1_z, 'tp_tipo_documento', 'string', c2_n, c2_v, c2_z, 'num_sequencial_alteracoes', 'int', c3_n, c3_v, c3_z, 'cod_evento', 'int', c4_n, c4_v, c4_z, 'cod_empresa', 'string', c5_n, c5_v, c5_z, 'nm_empresa', 'string', c6_n, c6_v, c6_z, 'cod_exercicio', 'string', c7_n, c7_v, c7_z, 'cod_unidade_negocios', 'string', c8_n, c8_v, c8_z, 'num_documento_inventario', 'string', c9_n, c9_v, c9_z, 'cod_requisicao_compra', 'string', c10_n, c10_v, c10_z, 'cod_item_requisicao_compra', 'string', c11_n, c11_v, c11_z, 'num_acompanhamento', 'string', c12_n, c12_v, c12_z, 'cod_classificacao_contabil_req', 'string', c13_n, c13_v, c13_z, 'cod_classe_custo_req', 'string', c14_n, c14_v, c14_z, 'num_ordem_req', 'string', c15_n, c15_v, c15_z, 'desc_texto_breve_req', 'string', c16_n, c16_v, c16_z, 'num_pedido_compras', 'string', c17_n, c17_v, c17_z, 'num_item_pedido_compras', 'string', c18_n, c18_v, c18_z, 'tp_tipo_pedido', 'string', c19_n, c19_v, c19_z, 'cod_classificacao_contabil', 'string', c20_n, c20_v, c20_z, 'cod_classe_custo', 'string', c21_n, c21_v, c21_z, 'cod_classificacao_contabil_ped', 'string', c22_n, c22_v, c22_z, 'cod_classe_custo_ped', 'string', c23_n, c23_v, c23_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`cod_gl_mm_data_da_aprovacao` IS NULL) AS c0_n, COUNT_IF(`cod_gl_mm_data_da_aprovacao` IS NOT NULL AND LOWER(TRIM(`cod_gl_mm_data_da_aprovacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c0_v, COUNT_IF(TRIM(`cod_gl_mm_data_da_aprovacao`) RLIKE '^0+([.,]0+)?$') AS c0_z, COUNT_IF(`num_documento` IS NULL) AS c1_n, COUNT_IF(`num_documento` IS NOT NULL AND LOWER(TRIM(`num_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c1_v, COUNT_IF(TRIM(`num_documento`) RLIKE '^0+([.,]0+)?$') AS c1_z, COUNT_IF(`tp_tipo_documento` IS NULL) AS c2_n, COUNT_IF(`tp_tipo_documento` IS NOT NULL AND LOWER(TRIM(`tp_tipo_documento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c2_v, COUNT_IF(TRIM(`tp_tipo_documento`) RLIKE '^0+([.,]0+)?$') AS c2_z, COUNT_IF(`num_sequencial_alteracoes` IS NULL) AS c3_n, CAST(0 AS BIGINT) AS c3_v, COUNT_IF(`num_sequencial_alteracoes` = 0) AS c3_z, COUNT_IF(`cod_evento` IS NULL) AS c4_n, CAST(0 AS BIGINT) AS c4_v, COUNT_IF(`cod_evento` = 0) AS c4_z, COUNT_IF(`cod_empresa` IS NULL) AS c5_n, COUNT_IF(`cod_empresa` IS NOT NULL AND LOWER(TRIM(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c5_v, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+([.,]0+)?$') AS c5_z, COUNT_IF(`nm_empresa` IS NULL) AS c6_n, COUNT_IF(`nm_empresa` IS NOT NULL AND LOWER(TRIM(`nm_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c6_v, COUNT_IF(TRIM(`nm_empresa`) RLIKE '^0+([.,]0+)?$') AS c6_z, COUNT_IF(`cod_exercicio` IS NULL) AS c7_n, COUNT_IF(`cod_exercicio` IS NOT NULL AND LOWER(TRIM(`cod_exercicio`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c7_v, COUNT_IF(TRIM(`cod_exercicio`) RLIKE '^0+([.,]0+)?$') AS c7_z, COUNT_IF(`cod_unidade_negocios` IS NULL) AS c8_n, COUNT_IF(`cod_unidade_negocios` IS NOT NULL AND LOWER(TRIM(`cod_unidade_negocios`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c8_v, COUNT_IF(TRIM(`cod_unidade_negocios`) RLIKE '^0+([.,]0+)?$') AS c8_z, COUNT_IF(`num_documento_inventario` IS NULL) AS c9_n, COUNT_IF(`num_documento_inventario` IS NOT NULL AND LOWER(TRIM(`num_documento_inventario`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c9_v, COUNT_IF(TRIM(`num_documento_inventario`) RLIKE '^0+([.,]0+)?$') AS c9_z, COUNT_IF(`cod_requisicao_compra` IS NULL) AS c10_n, COUNT_IF(`cod_requisicao_compra` IS NOT NULL AND LOWER(TRIM(`cod_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c10_v, COUNT_IF(TRIM(`cod_requisicao_compra`) RLIKE '^0+([.,]0+)?$') AS c10_z, COUNT_IF(`cod_item_requisicao_compra` IS NULL) AS c11_n, COUNT_IF(`cod_item_requisicao_compra` IS NOT NULL AND LOWER(TRIM(`cod_item_requisicao_compra`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c11_v, COUNT_IF(TRIM(`cod_item_requisicao_compra`) RLIKE '^0+([.,]0+)?$') AS c11_z, COUNT_IF(`num_acompanhamento` IS NULL) AS c12_n, COUNT_IF(`num_acompanhamento` IS NOT NULL AND LOWER(TRIM(`num_acompanhamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c12_v, COUNT_IF(TRIM(`num_acompanhamento`) RLIKE '^0+([.,]0+)?$') AS c12_z, COUNT_IF(`cod_classificacao_contabil_req` IS NULL) AS c13_n, COUNT_IF(`cod_classificacao_contabil_req` IS NOT NULL AND LOWER(TRIM(`cod_classificacao_contabil_req`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c13_v, COUNT_IF(TRIM(`cod_classificacao_contabil_req`) RLIKE '^0+([.,]0+)?$') AS c13_z, COUNT_IF(`cod_classe_custo_req` IS NULL) AS c14_n, COUNT_IF(`cod_classe_custo_req` IS NOT NULL AND LOWER(TRIM(`cod_classe_custo_req`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c14_v, COUNT_IF(TRIM(`cod_classe_custo_req`) RLIKE '^0+([.,]0+)?$') AS c14_z, COUNT_IF(`num_ordem_req` IS NULL) AS c15_n, COUNT_IF(`num_ordem_req` IS NOT NULL AND LOWER(TRIM(`num_ordem_req`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c15_v, COUNT_IF(TRIM(`num_ordem_req`) RLIKE '^0+([.,]0+)?$') AS c15_z, COUNT_IF(`desc_texto_breve_req` IS NULL) AS c16_n, COUNT_IF(`desc_texto_breve_req` IS NOT NULL AND LOWER(TRIM(`desc_texto_breve_req`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c16_v, COUNT_IF(TRIM(`desc_texto_breve_req`) RLIKE '^0+([.,]0+)?$') AS c16_z, COUNT_IF(`num_pedido_compras` IS NULL) AS c17_n, COUNT_IF(`num_pedido_compras` IS NOT NULL AND LOWER(TRIM(`num_pedido_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c17_v, COUNT_IF(TRIM(`num_pedido_compras`) RLIKE '^0+([.,]0+)?$') AS c17_z, COUNT_IF(`num_item_pedido_compras` IS NULL) AS c18_n, COUNT_IF(`num_item_pedido_compras` IS NOT NULL AND LOWER(TRIM(`num_item_pedido_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c18_v, COUNT_IF(TRIM(`num_item_pedido_compras`) RLIKE '^0+([.,]0+)?$') AS c18_z, COUNT_IF(`tp_tipo_pedido` IS NULL) AS c19_n, COUNT_IF(`tp_tipo_pedido` IS NOT NULL AND LOWER(TRIM(`tp_tipo_pedido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c19_v, COUNT_IF(TRIM(`tp_tipo_pedido`) RLIKE '^0+([.,]0+)?$') AS c19_z, COUNT_IF(`cod_classificacao_contabil` IS NULL) AS c20_n, COUNT_IF(`cod_classificacao_contabil` IS NOT NULL AND LOWER(TRIM(`cod_classificacao_contabil`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c20_v, COUNT_IF(TRIM(`cod_classificacao_contabil`) RLIKE '^0+([.,]0+)?$') AS c20_z, COUNT_IF(`cod_classe_custo` IS NULL) AS c21_n, COUNT_IF(`cod_classe_custo` IS NOT NULL AND LOWER(TRIM(`cod_classe_custo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c21_v, COUNT_IF(TRIM(`cod_classe_custo`) RLIKE '^0+([.,]0+)?$') AS c21_z, COUNT_IF(`cod_classificacao_contabil_ped` IS NULL) AS c22_n, COUNT_IF(`cod_classificacao_contabil_ped` IS NOT NULL AND LOWER(TRIM(`cod_classificacao_contabil_ped`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c22_v, COUNT_IF(TRIM(`cod_classificacao_contabil_ped`) RLIKE '^0+([.,]0+)?$') AS c22_z, COUNT_IF(`cod_classe_custo_ped` IS NULL) AS c23_n, COUNT_IF(`cod_classe_custo_ped` IS NOT NULL AND LOWER(TRIM(`cod_classe_custo_ped`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c23_v, COUNT_IF(TRIM(`cod_classe_custo_ped`) RLIKE '^0+([.,]0+)?$') AS c23_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'num_ordem_ped', 'string', c24_n, c24_v, c24_z, 'desc_texto_breve_ped', 'string', c25_n, c25_v, c25_z, 'num_ordem', 'string', c26_n, c26_v, c26_z, 'num_ordem_interna', 'string', c27_n, c27_v, c27_z, 'cod_categoria_item', 'string', c28_n, c28_v, c28_z, 'cod_elemento_pep', 'string', c29_n, c29_v, c29_z, 'num_contrato', 'string', c30_n, c30_v, c30_z, 'num_item_contrato', 'string', c31_n, c31_v, c31_z, 'num_contrato_re', 'string', c32_n, c32_v, c32_z, 'num_item_contrato_re', 'string', c33_n, c33_v, c33_z, 'cod_fornecedor', 'string', c34_n, c34_v, c34_z, 'cod_fornecedor_re', 'string', c35_n, c35_v, c35_z, 'nm_fornecedor_re', 'string', c36_n, c36_v, c36_z, 'num_material', 'string', c37_n, c37_v, c37_z, 'desc_texto_breve', 'string', c38_n, c38_v, c38_z, 'num_folha_servicos', 'string', c39_n, c39_v, c39_z, 'num_servico', 'string', c40_n, c40_v, c40_z, 'num_servico_re', 'string', c41_n, c41_v, c41_z, 'sg_unidade_medida', 'string', c42_n, c42_v, c42_z, 'sg_unidade_medida_base', 'string', c43_n, c43_v, c43_z, 'cod_centro', 'string', c44_n, c44_v, c44_z, 'cod_centro_ped', 'string', c45_n, c45_v, c45_z, 'cod_centro_custo', 'string', c46_n, c46_v, c46_z, 'desc_denominacao', 'string', c47_n, c47_v, c47_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`num_ordem_ped` IS NULL) AS c24_n, COUNT_IF(`num_ordem_ped` IS NOT NULL AND LOWER(TRIM(`num_ordem_ped`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c24_v, COUNT_IF(TRIM(`num_ordem_ped`) RLIKE '^0+([.,]0+)?$') AS c24_z, COUNT_IF(`desc_texto_breve_ped` IS NULL) AS c25_n, COUNT_IF(`desc_texto_breve_ped` IS NOT NULL AND LOWER(TRIM(`desc_texto_breve_ped`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c25_v, COUNT_IF(TRIM(`desc_texto_breve_ped`) RLIKE '^0+([.,]0+)?$') AS c25_z, COUNT_IF(`num_ordem` IS NULL) AS c26_n, COUNT_IF(`num_ordem` IS NOT NULL AND LOWER(TRIM(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c26_v, COUNT_IF(TRIM(`num_ordem`) RLIKE '^0+([.,]0+)?$') AS c26_z, COUNT_IF(`num_ordem_interna` IS NULL) AS c27_n, COUNT_IF(`num_ordem_interna` IS NOT NULL AND LOWER(TRIM(`num_ordem_interna`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c27_v, COUNT_IF(TRIM(`num_ordem_interna`) RLIKE '^0+([.,]0+)?$') AS c27_z, COUNT_IF(`cod_categoria_item` IS NULL) AS c28_n, COUNT_IF(`cod_categoria_item` IS NOT NULL AND LOWER(TRIM(`cod_categoria_item`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c28_v, COUNT_IF(TRIM(`cod_categoria_item`) RLIKE '^0+([.,]0+)?$') AS c28_z, COUNT_IF(`cod_elemento_pep` IS NULL) AS c29_n, COUNT_IF(`cod_elemento_pep` IS NOT NULL AND LOWER(TRIM(`cod_elemento_pep`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c29_v, COUNT_IF(TRIM(`cod_elemento_pep`) RLIKE '^0+([.,]0+)?$') AS c29_z, COUNT_IF(`num_contrato` IS NULL) AS c30_n, COUNT_IF(`num_contrato` IS NOT NULL AND LOWER(TRIM(`num_contrato`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c30_v, COUNT_IF(TRIM(`num_contrato`) RLIKE '^0+([.,]0+)?$') AS c30_z, COUNT_IF(`num_item_contrato` IS NULL) AS c31_n, COUNT_IF(`num_item_contrato` IS NOT NULL AND LOWER(TRIM(`num_item_contrato`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c31_v, COUNT_IF(TRIM(`num_item_contrato`) RLIKE '^0+([.,]0+)?$') AS c31_z, COUNT_IF(`num_contrato_re` IS NULL) AS c32_n, COUNT_IF(`num_contrato_re` IS NOT NULL AND LOWER(TRIM(`num_contrato_re`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c32_v, COUNT_IF(TRIM(`num_contrato_re`) RLIKE '^0+([.,]0+)?$') AS c32_z, COUNT_IF(`num_item_contrato_re` IS NULL) AS c33_n, COUNT_IF(`num_item_contrato_re` IS NOT NULL AND LOWER(TRIM(`num_item_contrato_re`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c33_v, COUNT_IF(TRIM(`num_item_contrato_re`) RLIKE '^0+([.,]0+)?$') AS c33_z, COUNT_IF(`cod_fornecedor` IS NULL) AS c34_n, COUNT_IF(`cod_fornecedor` IS NOT NULL AND LOWER(TRIM(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c34_v, COUNT_IF(TRIM(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$') AS c34_z, COUNT_IF(`cod_fornecedor_re` IS NULL) AS c35_n, COUNT_IF(`cod_fornecedor_re` IS NOT NULL AND LOWER(TRIM(`cod_fornecedor_re`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c35_v, COUNT_IF(TRIM(`cod_fornecedor_re`) RLIKE '^0+([.,]0+)?$') AS c35_z, COUNT_IF(`nm_fornecedor_re` IS NULL) AS c36_n, COUNT_IF(`nm_fornecedor_re` IS NOT NULL AND LOWER(TRIM(`nm_fornecedor_re`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c36_v, COUNT_IF(TRIM(`nm_fornecedor_re`) RLIKE '^0+([.,]0+)?$') AS c36_z, COUNT_IF(`num_material` IS NULL) AS c37_n, COUNT_IF(`num_material` IS NOT NULL AND LOWER(TRIM(`num_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c37_v, COUNT_IF(TRIM(`num_material`) RLIKE '^0+([.,]0+)?$') AS c37_z, COUNT_IF(`desc_texto_breve` IS NULL) AS c38_n, COUNT_IF(`desc_texto_breve` IS NOT NULL AND LOWER(TRIM(`desc_texto_breve`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c38_v, COUNT_IF(TRIM(`desc_texto_breve`) RLIKE '^0+([.,]0+)?$') AS c38_z, COUNT_IF(`num_folha_servicos` IS NULL) AS c39_n, COUNT_IF(`num_folha_servicos` IS NOT NULL AND LOWER(TRIM(`num_folha_servicos`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c39_v, COUNT_IF(TRIM(`num_folha_servicos`) RLIKE '^0+([.,]0+)?$') AS c39_z, COUNT_IF(`num_servico` IS NULL) AS c40_n, COUNT_IF(`num_servico` IS NOT NULL AND LOWER(TRIM(`num_servico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c40_v, COUNT_IF(TRIM(`num_servico`) RLIKE '^0+([.,]0+)?$') AS c40_z, COUNT_IF(`num_servico_re` IS NULL) AS c41_n, COUNT_IF(`num_servico_re` IS NOT NULL AND LOWER(TRIM(`num_servico_re`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c41_v, COUNT_IF(TRIM(`num_servico_re`) RLIKE '^0+([.,]0+)?$') AS c41_z, COUNT_IF(`sg_unidade_medida` IS NULL) AS c42_n, COUNT_IF(`sg_unidade_medida` IS NOT NULL AND LOWER(TRIM(`sg_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c42_v, COUNT_IF(TRIM(`sg_unidade_medida`) RLIKE '^0+([.,]0+)?$') AS c42_z, COUNT_IF(`sg_unidade_medida_base` IS NULL) AS c43_n, COUNT_IF(`sg_unidade_medida_base` IS NOT NULL AND LOWER(TRIM(`sg_unidade_medida_base`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c43_v, COUNT_IF(TRIM(`sg_unidade_medida_base`) RLIKE '^0+([.,]0+)?$') AS c43_z, COUNT_IF(`cod_centro` IS NULL) AS c44_n, COUNT_IF(`cod_centro` IS NOT NULL AND LOWER(TRIM(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c44_v, COUNT_IF(TRIM(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS c44_z, COUNT_IF(`cod_centro_ped` IS NULL) AS c45_n, COUNT_IF(`cod_centro_ped` IS NOT NULL AND LOWER(TRIM(`cod_centro_ped`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c45_v, COUNT_IF(TRIM(`cod_centro_ped`) RLIKE '^0+([.,]0+)?$') AS c45_z, COUNT_IF(`cod_centro_custo` IS NULL) AS c46_n, COUNT_IF(`cod_centro_custo` IS NOT NULL AND LOWER(TRIM(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c46_v, COUNT_IF(TRIM(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$') AS c46_z, COUNT_IF(`desc_denominacao` IS NULL) AS c47_n, COUNT_IF(`desc_denominacao` IS NOT NULL AND LOWER(TRIM(`desc_denominacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c47_v, COUNT_IF(TRIM(`desc_denominacao`) RLIKE '^0+([.,]0+)?$') AS c47_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'desc_texto_breve_re', 'string', c48_n, c48_v, c48_z, 'num_local_instalacao', 'string', c49_n, c49_v, c49_z, 'num_equipamento', 'string', c50_n, c50_v, c50_z, 'cod_org_compras', 'string', c51_n, c51_v, c51_z, 'cod_grupo_compradores', 'string', c52_n, c52_v, c52_z, 'vl_valor_bruto', 'double', c53_n, c53_v, c53_z, 'vl_limite_alcada', 'decimal(17,4)', c54_n, c54_v, c54_z, 'qt_quantidade', 'decimal(13,3)', c55_n, c55_v, c55_z, 'vl_fixado_cabecalho', 'double', c56_n, c56_v, c56_z, 'vl_preco_liquido_ped', 'double', c57_n, c57_v, c57_z, 'vl_preco_bruto', 'double', c58_n, c58_v, c58_z, 'sg_moeda', 'string', c59_n, c59_v, c59_z, 'sg_moeda_ped', 'string', c60_n, c60_v, c60_z, 'cod_estrategia_liberacao', 'string', c61_n, c61_v, c61_z, 'cod_status_liberacao', 'string', c62_n, c62_v, c62_z, 'cod_grupo_liberacao', 'string', c63_n, c63_v, c63_z, 'cod_grupo_liberador', 'string', c64_n, c64_v, c64_z, 'cod_liberacao', 'string', c65_n, c65_v, c65_z, 'cod_nivel_alcada', 'string', c66_n, c66_v, c66_z, 'cod_meio_aprovacao', 'string', c67_n, c67_v, c67_z, 'tp_movimento', 'string', c68_n, c68_v, c68_z, 'desc_cod_liberacao_01', 'string', c69_n, c69_v, c69_z, 'desc_cod_liberacao_02', 'string', c70_n, c70_v, c70_z, 'desc_cod_liberacao_03', 'string', c71_n, c71_v, c71_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`desc_texto_breve_re` IS NULL) AS c48_n, COUNT_IF(`desc_texto_breve_re` IS NOT NULL AND LOWER(TRIM(`desc_texto_breve_re`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c48_v, COUNT_IF(TRIM(`desc_texto_breve_re`) RLIKE '^0+([.,]0+)?$') AS c48_z, COUNT_IF(`num_local_instalacao` IS NULL) AS c49_n, COUNT_IF(`num_local_instalacao` IS NOT NULL AND LOWER(TRIM(`num_local_instalacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c49_v, COUNT_IF(TRIM(`num_local_instalacao`) RLIKE '^0+([.,]0+)?$') AS c49_z, COUNT_IF(`num_equipamento` IS NULL) AS c50_n, COUNT_IF(`num_equipamento` IS NOT NULL AND LOWER(TRIM(`num_equipamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c50_v, COUNT_IF(TRIM(`num_equipamento`) RLIKE '^0+([.,]0+)?$') AS c50_z, COUNT_IF(`cod_org_compras` IS NULL) AS c51_n, COUNT_IF(`cod_org_compras` IS NOT NULL AND LOWER(TRIM(`cod_org_compras`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c51_v, COUNT_IF(TRIM(`cod_org_compras`) RLIKE '^0+([.,]0+)?$') AS c51_z, COUNT_IF(`cod_grupo_compradores` IS NULL) AS c52_n, COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND LOWER(TRIM(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c52_v, COUNT_IF(TRIM(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$') AS c52_z, COUNT_IF(`vl_valor_bruto` IS NULL) AS c53_n, CAST(0 AS BIGINT) AS c53_v, COUNT_IF(`vl_valor_bruto` = 0) AS c53_z, COUNT_IF(`vl_limite_alcada` IS NULL) AS c54_n, CAST(0 AS BIGINT) AS c54_v, COUNT_IF(`vl_limite_alcada` = 0) AS c54_z, COUNT_IF(`qt_quantidade` IS NULL) AS c55_n, CAST(0 AS BIGINT) AS c55_v, COUNT_IF(`qt_quantidade` = 0) AS c55_z, COUNT_IF(`vl_fixado_cabecalho` IS NULL) AS c56_n, CAST(0 AS BIGINT) AS c56_v, COUNT_IF(`vl_fixado_cabecalho` = 0) AS c56_z, COUNT_IF(`vl_preco_liquido_ped` IS NULL) AS c57_n, CAST(0 AS BIGINT) AS c57_v, COUNT_IF(`vl_preco_liquido_ped` = 0) AS c57_z, COUNT_IF(`vl_preco_bruto` IS NULL) AS c58_n, CAST(0 AS BIGINT) AS c58_v, COUNT_IF(`vl_preco_bruto` = 0) AS c58_z, COUNT_IF(`sg_moeda` IS NULL) AS c59_n, COUNT_IF(`sg_moeda` IS NOT NULL AND LOWER(TRIM(`sg_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c59_v, COUNT_IF(TRIM(`sg_moeda`) RLIKE '^0+([.,]0+)?$') AS c59_z, COUNT_IF(`sg_moeda_ped` IS NULL) AS c60_n, COUNT_IF(`sg_moeda_ped` IS NOT NULL AND LOWER(TRIM(`sg_moeda_ped`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c60_v, COUNT_IF(TRIM(`sg_moeda_ped`) RLIKE '^0+([.,]0+)?$') AS c60_z, COUNT_IF(`cod_estrategia_liberacao` IS NULL) AS c61_n, COUNT_IF(`cod_estrategia_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_estrategia_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c61_v, COUNT_IF(TRIM(`cod_estrategia_liberacao`) RLIKE '^0+([.,]0+)?$') AS c61_z, COUNT_IF(`cod_status_liberacao` IS NULL) AS c62_n, COUNT_IF(`cod_status_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_status_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c62_v, COUNT_IF(TRIM(`cod_status_liberacao`) RLIKE '^0+([.,]0+)?$') AS c62_z, COUNT_IF(`cod_grupo_liberacao` IS NULL) AS c63_n, COUNT_IF(`cod_grupo_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_grupo_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c63_v, COUNT_IF(TRIM(`cod_grupo_liberacao`) RLIKE '^0+([.,]0+)?$') AS c63_z, COUNT_IF(`cod_grupo_liberador` IS NULL) AS c64_n, COUNT_IF(`cod_grupo_liberador` IS NOT NULL AND LOWER(TRIM(`cod_grupo_liberador`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c64_v, COUNT_IF(TRIM(`cod_grupo_liberador`) RLIKE '^0+([.,]0+)?$') AS c64_z, COUNT_IF(`cod_liberacao` IS NULL) AS c65_n, COUNT_IF(`cod_liberacao` IS NOT NULL AND LOWER(TRIM(`cod_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c65_v, COUNT_IF(TRIM(`cod_liberacao`) RLIKE '^0+([.,]0+)?$') AS c65_z, COUNT_IF(`cod_nivel_alcada` IS NULL) AS c66_n, COUNT_IF(`cod_nivel_alcada` IS NOT NULL AND LOWER(TRIM(`cod_nivel_alcada`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c66_v, COUNT_IF(TRIM(`cod_nivel_alcada`) RLIKE '^0+([.,]0+)?$') AS c66_z, COUNT_IF(`cod_meio_aprovacao` IS NULL) AS c67_n, COUNT_IF(`cod_meio_aprovacao` IS NOT NULL AND LOWER(TRIM(`cod_meio_aprovacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c67_v, COUNT_IF(TRIM(`cod_meio_aprovacao`) RLIKE '^0+([.,]0+)?$') AS c67_z, COUNT_IF(`tp_movimento` IS NULL) AS c68_n, COUNT_IF(`tp_movimento` IS NOT NULL AND LOWER(TRIM(`tp_movimento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c68_v, COUNT_IF(TRIM(`tp_movimento`) RLIKE '^0+([.,]0+)?$') AS c68_z, COUNT_IF(`desc_cod_liberacao_01` IS NULL) AS c69_n, COUNT_IF(`desc_cod_liberacao_01` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c69_v, COUNT_IF(TRIM(`desc_cod_liberacao_01`) RLIKE '^0+([.,]0+)?$') AS c69_z, COUNT_IF(`desc_cod_liberacao_02` IS NULL) AS c70_n, COUNT_IF(`desc_cod_liberacao_02` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c70_v, COUNT_IF(TRIM(`desc_cod_liberacao_02`) RLIKE '^0+([.,]0+)?$') AS c70_z, COUNT_IF(`desc_cod_liberacao_03` IS NULL) AS c71_n, COUNT_IF(`desc_cod_liberacao_03` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c71_v, COUNT_IF(TRIM(`desc_cod_liberacao_03`) RLIKE '^0+([.,]0+)?$') AS c71_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'desc_cod_liberacao_04', 'string', c72_n, c72_v, c72_z, 'desc_cod_liberacao_05', 'string', c73_n, c73_v, c73_z, 'desc_cod_liberacao_06', 'string', c74_n, c74_v, c74_z, 'desc_cod_liberacao_07', 'string', c75_n, c75_v, c75_z, 'desc_cod_liberacao_08', 'string', c76_n, c76_v, c76_z, 'cod_cadeia_aprovacao_n1', 'string', c77_n, c77_v, c77_z, 'cod_cadeia_aprovacao_n2', 'string', c78_n, c78_v, c78_z, 'cod_cadeia_aprovacao_n3', 'string', c79_n, c79_v, c79_z, 'cod_cadeia_aprovacao_n4', 'string', c80_n, c80_v, c80_z, 'cod_cadeia_aprovacao_n5', 'string', c81_n, c81_v, c81_z, 'cod_cadeia_aprovacao_n6', 'string', c82_n, c82_v, c82_z, 'cod_cadeia_aprovacao_n7', 'string', c83_n, c83_v, c83_z, 'cod_cadeia_aprovacao_n8', 'string', c84_n, c84_v, c84_z, 'cod_nivel_n1', 'string', c85_n, c85_v, c85_z, 'cod_nivel_n2', 'string', c86_n, c86_v, c86_z, 'cod_nivel_n3', 'string', c87_n, c87_v, c87_z, 'cod_nivel_n4', 'string', c88_n, c88_v, c88_z, 'cod_nivel_n5', 'string', c89_n, c89_v, c89_z, 'cod_nivel_n6', 'string', c90_n, c90_v, c90_z, 'cod_nivel_n7', 'string', c91_n, c91_v, c91_z, 'cod_nivel_n8', 'string', c92_n, c92_v, c92_z, 'nm_aprovador_01', 'string', c93_n, c93_v, c93_z, 'nm_aprovador_02', 'string', c94_n, c94_v, c94_z, 'nm_aprovador_03', 'string', c95_n, c95_v, c95_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`desc_cod_liberacao_04` IS NULL) AS c72_n, COUNT_IF(`desc_cod_liberacao_04` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c72_v, COUNT_IF(TRIM(`desc_cod_liberacao_04`) RLIKE '^0+([.,]0+)?$') AS c72_z, COUNT_IF(`desc_cod_liberacao_05` IS NULL) AS c73_n, COUNT_IF(`desc_cod_liberacao_05` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c73_v, COUNT_IF(TRIM(`desc_cod_liberacao_05`) RLIKE '^0+([.,]0+)?$') AS c73_z, COUNT_IF(`desc_cod_liberacao_06` IS NULL) AS c74_n, COUNT_IF(`desc_cod_liberacao_06` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c74_v, COUNT_IF(TRIM(`desc_cod_liberacao_06`) RLIKE '^0+([.,]0+)?$') AS c74_z, COUNT_IF(`desc_cod_liberacao_07` IS NULL) AS c75_n, COUNT_IF(`desc_cod_liberacao_07` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c75_v, COUNT_IF(TRIM(`desc_cod_liberacao_07`) RLIKE '^0+([.,]0+)?$') AS c75_z, COUNT_IF(`desc_cod_liberacao_08` IS NULL) AS c76_n, COUNT_IF(`desc_cod_liberacao_08` IS NOT NULL AND LOWER(TRIM(`desc_cod_liberacao_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c76_v, COUNT_IF(TRIM(`desc_cod_liberacao_08`) RLIKE '^0+([.,]0+)?$') AS c76_z, COUNT_IF(`cod_cadeia_aprovacao_n1` IS NULL) AS c77_n, COUNT_IF(`cod_cadeia_aprovacao_n1` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c77_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n1`) RLIKE '^0+([.,]0+)?$') AS c77_z, COUNT_IF(`cod_cadeia_aprovacao_n2` IS NULL) AS c78_n, COUNT_IF(`cod_cadeia_aprovacao_n2` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c78_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n2`) RLIKE '^0+([.,]0+)?$') AS c78_z, COUNT_IF(`cod_cadeia_aprovacao_n3` IS NULL) AS c79_n, COUNT_IF(`cod_cadeia_aprovacao_n3` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n3`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c79_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n3`) RLIKE '^0+([.,]0+)?$') AS c79_z, COUNT_IF(`cod_cadeia_aprovacao_n4` IS NULL) AS c80_n, COUNT_IF(`cod_cadeia_aprovacao_n4` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n4`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c80_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n4`) RLIKE '^0+([.,]0+)?$') AS c80_z, COUNT_IF(`cod_cadeia_aprovacao_n5` IS NULL) AS c81_n, COUNT_IF(`cod_cadeia_aprovacao_n5` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n5`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c81_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n5`) RLIKE '^0+([.,]0+)?$') AS c81_z, COUNT_IF(`cod_cadeia_aprovacao_n6` IS NULL) AS c82_n, COUNT_IF(`cod_cadeia_aprovacao_n6` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n6`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c82_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n6`) RLIKE '^0+([.,]0+)?$') AS c82_z, COUNT_IF(`cod_cadeia_aprovacao_n7` IS NULL) AS c83_n, COUNT_IF(`cod_cadeia_aprovacao_n7` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n7`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c83_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n7`) RLIKE '^0+([.,]0+)?$') AS c83_z, COUNT_IF(`cod_cadeia_aprovacao_n8` IS NULL) AS c84_n, COUNT_IF(`cod_cadeia_aprovacao_n8` IS NOT NULL AND LOWER(TRIM(`cod_cadeia_aprovacao_n8`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c84_v, COUNT_IF(TRIM(`cod_cadeia_aprovacao_n8`) RLIKE '^0+([.,]0+)?$') AS c84_z, COUNT_IF(`cod_nivel_n1` IS NULL) AS c85_n, COUNT_IF(`cod_nivel_n1` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n1`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c85_v, COUNT_IF(TRIM(`cod_nivel_n1`) RLIKE '^0+([.,]0+)?$') AS c85_z, COUNT_IF(`cod_nivel_n2` IS NULL) AS c86_n, COUNT_IF(`cod_nivel_n2` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n2`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c86_v, COUNT_IF(TRIM(`cod_nivel_n2`) RLIKE '^0+([.,]0+)?$') AS c86_z, COUNT_IF(`cod_nivel_n3` IS NULL) AS c87_n, COUNT_IF(`cod_nivel_n3` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n3`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c87_v, COUNT_IF(TRIM(`cod_nivel_n3`) RLIKE '^0+([.,]0+)?$') AS c87_z, COUNT_IF(`cod_nivel_n4` IS NULL) AS c88_n, COUNT_IF(`cod_nivel_n4` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n4`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c88_v, COUNT_IF(TRIM(`cod_nivel_n4`) RLIKE '^0+([.,]0+)?$') AS c88_z, COUNT_IF(`cod_nivel_n5` IS NULL) AS c89_n, COUNT_IF(`cod_nivel_n5` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n5`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c89_v, COUNT_IF(TRIM(`cod_nivel_n5`) RLIKE '^0+([.,]0+)?$') AS c89_z, COUNT_IF(`cod_nivel_n6` IS NULL) AS c90_n, COUNT_IF(`cod_nivel_n6` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n6`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c90_v, COUNT_IF(TRIM(`cod_nivel_n6`) RLIKE '^0+([.,]0+)?$') AS c90_z, COUNT_IF(`cod_nivel_n7` IS NULL) AS c91_n, COUNT_IF(`cod_nivel_n7` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n7`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c91_v, COUNT_IF(TRIM(`cod_nivel_n7`) RLIKE '^0+([.,]0+)?$') AS c91_z, COUNT_IF(`cod_nivel_n8` IS NULL) AS c92_n, COUNT_IF(`cod_nivel_n8` IS NOT NULL AND LOWER(TRIM(`cod_nivel_n8`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c92_v, COUNT_IF(TRIM(`cod_nivel_n8`) RLIKE '^0+([.,]0+)?$') AS c92_z, COUNT_IF(`nm_aprovador_01` IS NULL) AS c93_n, COUNT_IF(`nm_aprovador_01` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c93_v, COUNT_IF(TRIM(`nm_aprovador_01`) RLIKE '^0+([.,]0+)?$') AS c93_z, COUNT_IF(`nm_aprovador_02` IS NULL) AS c94_n, COUNT_IF(`nm_aprovador_02` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c94_v, COUNT_IF(TRIM(`nm_aprovador_02`) RLIKE '^0+([.,]0+)?$') AS c94_z, COUNT_IF(`nm_aprovador_03` IS NULL) AS c95_n, COUNT_IF(`nm_aprovador_03` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c95_v, COUNT_IF(TRIM(`nm_aprovador_03`) RLIKE '^0+([.,]0+)?$') AS c95_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'nm_aprovador_04', 'string', c96_n, c96_v, c96_z, 'nm_aprovador_05', 'string', c97_n, c97_v, c97_z, 'nm_aprovador_06', 'string', c98_n, c98_v, c98_z, 'nm_aprovador_07', 'string', c99_n, c99_v, c99_z, 'nm_aprovador_08', 'string', c100_n, c100_v, c100_z, 'cod_liberacao_01', 'string', c101_n, c101_v, c101_z, 'cod_liberacao_02', 'string', c102_n, c102_v, c102_z, 'cod_liberacao_03', 'string', c103_n, c103_v, c103_z, 'cod_liberacao_04', 'string', c104_n, c104_v, c104_z, 'cod_liberacao_05', 'string', c105_n, c105_v, c105_z, 'cod_liberacao_06', 'string', c106_n, c106_v, c106_z, 'cod_liberacao_07', 'string', c107_n, c107_v, c107_z, 'cod_liberacao_08', 'string', c108_n, c108_v, c108_z, 'cod_usuario_lib_01', 'string', c109_n, c109_v, c109_z, 'cod_usuario_lib_02', 'string', c110_n, c110_v, c110_z, 'cod_usuario_lib_03', 'string', c111_n, c111_v, c111_z, 'cod_usuario_lib_04', 'string', c112_n, c112_v, c112_z, 'cod_usuario_lib_05', 'string', c113_n, c113_v, c113_z, 'cod_usuario_lib_06', 'string', c114_n, c114_v, c114_z, 'cod_usuario_lib_07', 'string', c115_n, c115_v, c115_z, 'cod_usuario_lib_08', 'string', c116_n, c116_v, c116_z, 'dt_recebimento_01', 'date', c117_n, c117_v, c117_z, 'dt_recebimento_02', 'date', c118_n, c118_v, c118_z, 'dt_recebimento_03', 'date', c119_n, c119_v, c119_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`nm_aprovador_04` IS NULL) AS c96_n, COUNT_IF(`nm_aprovador_04` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c96_v, COUNT_IF(TRIM(`nm_aprovador_04`) RLIKE '^0+([.,]0+)?$') AS c96_z, COUNT_IF(`nm_aprovador_05` IS NULL) AS c97_n, COUNT_IF(`nm_aprovador_05` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c97_v, COUNT_IF(TRIM(`nm_aprovador_05`) RLIKE '^0+([.,]0+)?$') AS c97_z, COUNT_IF(`nm_aprovador_06` IS NULL) AS c98_n, COUNT_IF(`nm_aprovador_06` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c98_v, COUNT_IF(TRIM(`nm_aprovador_06`) RLIKE '^0+([.,]0+)?$') AS c98_z, COUNT_IF(`nm_aprovador_07` IS NULL) AS c99_n, COUNT_IF(`nm_aprovador_07` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c99_v, COUNT_IF(TRIM(`nm_aprovador_07`) RLIKE '^0+([.,]0+)?$') AS c99_z, COUNT_IF(`nm_aprovador_08` IS NULL) AS c100_n, COUNT_IF(`nm_aprovador_08` IS NOT NULL AND LOWER(TRIM(`nm_aprovador_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c100_v, COUNT_IF(TRIM(`nm_aprovador_08`) RLIKE '^0+([.,]0+)?$') AS c100_z, COUNT_IF(`cod_liberacao_01` IS NULL) AS c101_n, COUNT_IF(`cod_liberacao_01` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c101_v, COUNT_IF(TRIM(`cod_liberacao_01`) RLIKE '^0+([.,]0+)?$') AS c101_z, COUNT_IF(`cod_liberacao_02` IS NULL) AS c102_n, COUNT_IF(`cod_liberacao_02` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c102_v, COUNT_IF(TRIM(`cod_liberacao_02`) RLIKE '^0+([.,]0+)?$') AS c102_z, COUNT_IF(`cod_liberacao_03` IS NULL) AS c103_n, COUNT_IF(`cod_liberacao_03` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c103_v, COUNT_IF(TRIM(`cod_liberacao_03`) RLIKE '^0+([.,]0+)?$') AS c103_z, COUNT_IF(`cod_liberacao_04` IS NULL) AS c104_n, COUNT_IF(`cod_liberacao_04` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c104_v, COUNT_IF(TRIM(`cod_liberacao_04`) RLIKE '^0+([.,]0+)?$') AS c104_z, COUNT_IF(`cod_liberacao_05` IS NULL) AS c105_n, COUNT_IF(`cod_liberacao_05` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c105_v, COUNT_IF(TRIM(`cod_liberacao_05`) RLIKE '^0+([.,]0+)?$') AS c105_z, COUNT_IF(`cod_liberacao_06` IS NULL) AS c106_n, COUNT_IF(`cod_liberacao_06` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c106_v, COUNT_IF(TRIM(`cod_liberacao_06`) RLIKE '^0+([.,]0+)?$') AS c106_z, COUNT_IF(`cod_liberacao_07` IS NULL) AS c107_n, COUNT_IF(`cod_liberacao_07` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c107_v, COUNT_IF(TRIM(`cod_liberacao_07`) RLIKE '^0+([.,]0+)?$') AS c107_z, COUNT_IF(`cod_liberacao_08` IS NULL) AS c108_n, COUNT_IF(`cod_liberacao_08` IS NOT NULL AND LOWER(TRIM(`cod_liberacao_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c108_v, COUNT_IF(TRIM(`cod_liberacao_08`) RLIKE '^0+([.,]0+)?$') AS c108_z, COUNT_IF(`cod_usuario_lib_01` IS NULL) AS c109_n, COUNT_IF(`cod_usuario_lib_01` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c109_v, COUNT_IF(TRIM(`cod_usuario_lib_01`) RLIKE '^0+([.,]0+)?$') AS c109_z, COUNT_IF(`cod_usuario_lib_02` IS NULL) AS c110_n, COUNT_IF(`cod_usuario_lib_02` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c110_v, COUNT_IF(TRIM(`cod_usuario_lib_02`) RLIKE '^0+([.,]0+)?$') AS c110_z, COUNT_IF(`cod_usuario_lib_03` IS NULL) AS c111_n, COUNT_IF(`cod_usuario_lib_03` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c111_v, COUNT_IF(TRIM(`cod_usuario_lib_03`) RLIKE '^0+([.,]0+)?$') AS c111_z, COUNT_IF(`cod_usuario_lib_04` IS NULL) AS c112_n, COUNT_IF(`cod_usuario_lib_04` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c112_v, COUNT_IF(TRIM(`cod_usuario_lib_04`) RLIKE '^0+([.,]0+)?$') AS c112_z, COUNT_IF(`cod_usuario_lib_05` IS NULL) AS c113_n, COUNT_IF(`cod_usuario_lib_05` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c113_v, COUNT_IF(TRIM(`cod_usuario_lib_05`) RLIKE '^0+([.,]0+)?$') AS c113_z, COUNT_IF(`cod_usuario_lib_06` IS NULL) AS c114_n, COUNT_IF(`cod_usuario_lib_06` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c114_v, COUNT_IF(TRIM(`cod_usuario_lib_06`) RLIKE '^0+([.,]0+)?$') AS c114_z, COUNT_IF(`cod_usuario_lib_07` IS NULL) AS c115_n, COUNT_IF(`cod_usuario_lib_07` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c115_v, COUNT_IF(TRIM(`cod_usuario_lib_07`) RLIKE '^0+([.,]0+)?$') AS c115_z, COUNT_IF(`cod_usuario_lib_08` IS NULL) AS c116_n, COUNT_IF(`cod_usuario_lib_08` IS NOT NULL AND LOWER(TRIM(`cod_usuario_lib_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c116_v, COUNT_IF(TRIM(`cod_usuario_lib_08`) RLIKE '^0+([.,]0+)?$') AS c116_z, COUNT_IF(`dt_recebimento_01` IS NULL) AS c117_n, CAST(0 AS BIGINT) AS c117_v, CAST(0 AS BIGINT) AS c117_z, COUNT_IF(`dt_recebimento_02` IS NULL) AS c118_n, CAST(0 AS BIGINT) AS c118_v, CAST(0 AS BIGINT) AS c118_z, COUNT_IF(`dt_recebimento_03` IS NULL) AS c119_n, CAST(0 AS BIGINT) AS c119_v, CAST(0 AS BIGINT) AS c119_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'dt_recebimento_04', 'date', c120_n, c120_v, c120_z, 'dt_recebimento_05', 'date', c121_n, c121_v, c121_z, 'dt_recebimento_06', 'date', c122_n, c122_v, c122_z, 'dt_recebimento_07', 'date', c123_n, c123_v, c123_z, 'dt_recebimento_08', 'date', c124_n, c124_v, c124_z, 'hr_recebimento_01', 'string', c125_n, c125_v, c125_z, 'hr_recebimento_02', 'string', c126_n, c126_v, c126_z, 'hr_recebimento_03', 'string', c127_n, c127_v, c127_z, 'hr_recebimento_04', 'string', c128_n, c128_v, c128_z, 'hr_recebimento_05', 'string', c129_n, c129_v, c129_z, 'hr_recebimento_06', 'string', c130_n, c130_v, c130_z, 'hr_recebimento_07', 'string', c131_n, c131_v, c131_z, 'hr_recebimento_08', 'string', c132_n, c132_v, c132_z, 'dt_recebimento_msg', 'date', c133_n, c133_v, c133_z, 'hr_recebimento_msg', 'string', c134_n, c134_v, c134_z, 'dt_liberacao_01', 'date', c135_n, c135_v, c135_z, 'dt_liberacao_02', 'date', c136_n, c136_v, c136_z, 'dt_liberacao_03', 'date', c137_n, c137_v, c137_z, 'dt_liberacao_04', 'date', c138_n, c138_v, c138_z, 'dt_liberacao_05', 'date', c139_n, c139_v, c139_z, 'dt_liberacao_06', 'date', c140_n, c140_v, c140_z, 'dt_liberacao_07', 'date', c141_n, c141_v, c141_z, 'dt_liberacao_08', 'date', c142_n, c142_v, c142_z, 'dt_liberacao_final', 'date', c143_n, c143_v, c143_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`dt_recebimento_04` IS NULL) AS c120_n, CAST(0 AS BIGINT) AS c120_v, CAST(0 AS BIGINT) AS c120_z, COUNT_IF(`dt_recebimento_05` IS NULL) AS c121_n, CAST(0 AS BIGINT) AS c121_v, CAST(0 AS BIGINT) AS c121_z, COUNT_IF(`dt_recebimento_06` IS NULL) AS c122_n, CAST(0 AS BIGINT) AS c122_v, CAST(0 AS BIGINT) AS c122_z, COUNT_IF(`dt_recebimento_07` IS NULL) AS c123_n, CAST(0 AS BIGINT) AS c123_v, CAST(0 AS BIGINT) AS c123_z, COUNT_IF(`dt_recebimento_08` IS NULL) AS c124_n, CAST(0 AS BIGINT) AS c124_v, CAST(0 AS BIGINT) AS c124_z, COUNT_IF(`hr_recebimento_01` IS NULL) AS c125_n, COUNT_IF(`hr_recebimento_01` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c125_v, COUNT_IF(TRIM(`hr_recebimento_01`) RLIKE '^0+([.,]0+)?$') AS c125_z, COUNT_IF(`hr_recebimento_02` IS NULL) AS c126_n, COUNT_IF(`hr_recebimento_02` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c126_v, COUNT_IF(TRIM(`hr_recebimento_02`) RLIKE '^0+([.,]0+)?$') AS c126_z, COUNT_IF(`hr_recebimento_03` IS NULL) AS c127_n, COUNT_IF(`hr_recebimento_03` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c127_v, COUNT_IF(TRIM(`hr_recebimento_03`) RLIKE '^0+([.,]0+)?$') AS c127_z, COUNT_IF(`hr_recebimento_04` IS NULL) AS c128_n, COUNT_IF(`hr_recebimento_04` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c128_v, COUNT_IF(TRIM(`hr_recebimento_04`) RLIKE '^0+([.,]0+)?$') AS c128_z, COUNT_IF(`hr_recebimento_05` IS NULL) AS c129_n, COUNT_IF(`hr_recebimento_05` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c129_v, COUNT_IF(TRIM(`hr_recebimento_05`) RLIKE '^0+([.,]0+)?$') AS c129_z, COUNT_IF(`hr_recebimento_06` IS NULL) AS c130_n, COUNT_IF(`hr_recebimento_06` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c130_v, COUNT_IF(TRIM(`hr_recebimento_06`) RLIKE '^0+([.,]0+)?$') AS c130_z, COUNT_IF(`hr_recebimento_07` IS NULL) AS c131_n, COUNT_IF(`hr_recebimento_07` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c131_v, COUNT_IF(TRIM(`hr_recebimento_07`) RLIKE '^0+([.,]0+)?$') AS c131_z, COUNT_IF(`hr_recebimento_08` IS NULL) AS c132_n, COUNT_IF(`hr_recebimento_08` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c132_v, COUNT_IF(TRIM(`hr_recebimento_08`) RLIKE '^0+([.,]0+)?$') AS c132_z, COUNT_IF(`dt_recebimento_msg` IS NULL) AS c133_n, CAST(0 AS BIGINT) AS c133_v, CAST(0 AS BIGINT) AS c133_z, COUNT_IF(`hr_recebimento_msg` IS NULL) AS c134_n, COUNT_IF(`hr_recebimento_msg` IS NOT NULL AND LOWER(TRIM(`hr_recebimento_msg`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c134_v, COUNT_IF(TRIM(`hr_recebimento_msg`) RLIKE '^0+([.,]0+)?$') AS c134_z, COUNT_IF(`dt_liberacao_01` IS NULL) AS c135_n, CAST(0 AS BIGINT) AS c135_v, CAST(0 AS BIGINT) AS c135_z, COUNT_IF(`dt_liberacao_02` IS NULL) AS c136_n, CAST(0 AS BIGINT) AS c136_v, CAST(0 AS BIGINT) AS c136_z, COUNT_IF(`dt_liberacao_03` IS NULL) AS c137_n, CAST(0 AS BIGINT) AS c137_v, CAST(0 AS BIGINT) AS c137_z, COUNT_IF(`dt_liberacao_04` IS NULL) AS c138_n, CAST(0 AS BIGINT) AS c138_v, CAST(0 AS BIGINT) AS c138_z, COUNT_IF(`dt_liberacao_05` IS NULL) AS c139_n, CAST(0 AS BIGINT) AS c139_v, CAST(0 AS BIGINT) AS c139_z, COUNT_IF(`dt_liberacao_06` IS NULL) AS c140_n, CAST(0 AS BIGINT) AS c140_v, CAST(0 AS BIGINT) AS c140_z, COUNT_IF(`dt_liberacao_07` IS NULL) AS c141_n, CAST(0 AS BIGINT) AS c141_v, CAST(0 AS BIGINT) AS c141_z, COUNT_IF(`dt_liberacao_08` IS NULL) AS c142_n, CAST(0 AS BIGINT) AS c142_v, CAST(0 AS BIGINT) AS c142_z, COUNT_IF(`dt_liberacao_final` IS NULL) AS c143_n, CAST(0 AS BIGINT) AS c143_v, CAST(0 AS BIGINT) AS c143_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'hr_liberacao_01', 'string', c144_n, c144_v, c144_z, 'hr_liberacao_02', 'string', c145_n, c145_v, c145_z, 'hr_liberacao_03', 'string', c146_n, c146_v, c146_z, 'hr_liberacao_04', 'string', c147_n, c147_v, c147_z, 'hr_liberacao_05', 'string', c148_n, c148_v, c148_z, 'hr_liberacao_06', 'string', c149_n, c149_v, c149_z, 'hr_liberacao_07', 'string', c150_n, c150_v, c150_z, 'hr_liberacao_08', 'string', c151_n, c151_v, c151_z, 'hr_liberacao_final', 'string', c152_n, c152_v, c152_z, 'hr_entrada', 'string', c153_n, c153_v, c153_z, 'dt_lancamento', 'date', c154_n, c154_v, c154_z, 'dt_remessa', 'date', c155_n, c155_v, c155_z, 'nm_requisitante', 'string', c156_n, c156_v, c156_z, 'nm_solicitante', 'string', c157_n, c157_v, c157_z, 'nm_usuario_aprovador', 'string', c158_n, c158_v, c158_z, 'cod_criado_por', 'string', c159_n, c159_v, c159_z, 'cod_usuario_substituido', 'string', c160_n, c160_v, c160_z, 'cod_proximo_aprovador', 'string', c161_n, c161_v, c161_z, 'cod_usuario_grupo_aprovador', 'string', c162_n, c162_v, c162_z, 'ind_delegacao_ativa', 'string', c163_n, c163_v, c163_z, 'cod_responsavel_delegacao', 'string', c164_n, c164_v, c164_z, 'dt_validade_delegacao', 'date', c165_n, c165_v, c165_z, 'cod_status_atual', 'string', c166_n, c166_v, c166_z, 'cod_status_portal', 'string', c167_n, c167_v, c167_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`hr_liberacao_01` IS NULL) AS c144_n, COUNT_IF(`hr_liberacao_01` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c144_v, COUNT_IF(TRIM(`hr_liberacao_01`) RLIKE '^0+([.,]0+)?$') AS c144_z, COUNT_IF(`hr_liberacao_02` IS NULL) AS c145_n, COUNT_IF(`hr_liberacao_02` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c145_v, COUNT_IF(TRIM(`hr_liberacao_02`) RLIKE '^0+([.,]0+)?$') AS c145_z, COUNT_IF(`hr_liberacao_03` IS NULL) AS c146_n, COUNT_IF(`hr_liberacao_03` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c146_v, COUNT_IF(TRIM(`hr_liberacao_03`) RLIKE '^0+([.,]0+)?$') AS c146_z, COUNT_IF(`hr_liberacao_04` IS NULL) AS c147_n, COUNT_IF(`hr_liberacao_04` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c147_v, COUNT_IF(TRIM(`hr_liberacao_04`) RLIKE '^0+([.,]0+)?$') AS c147_z, COUNT_IF(`hr_liberacao_05` IS NULL) AS c148_n, COUNT_IF(`hr_liberacao_05` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c148_v, COUNT_IF(TRIM(`hr_liberacao_05`) RLIKE '^0+([.,]0+)?$') AS c148_z, COUNT_IF(`hr_liberacao_06` IS NULL) AS c149_n, COUNT_IF(`hr_liberacao_06` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c149_v, COUNT_IF(TRIM(`hr_liberacao_06`) RLIKE '^0+([.,]0+)?$') AS c149_z, COUNT_IF(`hr_liberacao_07` IS NULL) AS c150_n, COUNT_IF(`hr_liberacao_07` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c150_v, COUNT_IF(TRIM(`hr_liberacao_07`) RLIKE '^0+([.,]0+)?$') AS c150_z, COUNT_IF(`hr_liberacao_08` IS NULL) AS c151_n, COUNT_IF(`hr_liberacao_08` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c151_v, COUNT_IF(TRIM(`hr_liberacao_08`) RLIKE '^0+([.,]0+)?$') AS c151_z, COUNT_IF(`hr_liberacao_final` IS NULL) AS c152_n, COUNT_IF(`hr_liberacao_final` IS NOT NULL AND LOWER(TRIM(`hr_liberacao_final`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c152_v, COUNT_IF(TRIM(`hr_liberacao_final`) RLIKE '^0+([.,]0+)?$') AS c152_z, COUNT_IF(`hr_entrada` IS NULL) AS c153_n, COUNT_IF(`hr_entrada` IS NOT NULL AND LOWER(TRIM(`hr_entrada`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c153_v, COUNT_IF(TRIM(`hr_entrada`) RLIKE '^0+([.,]0+)?$') AS c153_z, COUNT_IF(`dt_lancamento` IS NULL) AS c154_n, CAST(0 AS BIGINT) AS c154_v, CAST(0 AS BIGINT) AS c154_z, COUNT_IF(`dt_remessa` IS NULL) AS c155_n, CAST(0 AS BIGINT) AS c155_v, CAST(0 AS BIGINT) AS c155_z, COUNT_IF(`nm_requisitante` IS NULL) AS c156_n, COUNT_IF(`nm_requisitante` IS NOT NULL AND LOWER(TRIM(`nm_requisitante`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c156_v, COUNT_IF(TRIM(`nm_requisitante`) RLIKE '^0+([.,]0+)?$') AS c156_z, COUNT_IF(`nm_solicitante` IS NULL) AS c157_n, COUNT_IF(`nm_solicitante` IS NOT NULL AND LOWER(TRIM(`nm_solicitante`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c157_v, COUNT_IF(TRIM(`nm_solicitante`) RLIKE '^0+([.,]0+)?$') AS c157_z, COUNT_IF(`nm_usuario_aprovador` IS NULL) AS c158_n, COUNT_IF(`nm_usuario_aprovador` IS NOT NULL AND LOWER(TRIM(`nm_usuario_aprovador`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c158_v, COUNT_IF(TRIM(`nm_usuario_aprovador`) RLIKE '^0+([.,]0+)?$') AS c158_z, COUNT_IF(`cod_criado_por` IS NULL) AS c159_n, COUNT_IF(`cod_criado_por` IS NOT NULL AND LOWER(TRIM(`cod_criado_por`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c159_v, COUNT_IF(TRIM(`cod_criado_por`) RLIKE '^0+([.,]0+)?$') AS c159_z, COUNT_IF(`cod_usuario_substituido` IS NULL) AS c160_n, COUNT_IF(`cod_usuario_substituido` IS NOT NULL AND LOWER(TRIM(`cod_usuario_substituido`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c160_v, COUNT_IF(TRIM(`cod_usuario_substituido`) RLIKE '^0+([.,]0+)?$') AS c160_z, COUNT_IF(`cod_proximo_aprovador` IS NULL) AS c161_n, COUNT_IF(`cod_proximo_aprovador` IS NOT NULL AND LOWER(TRIM(`cod_proximo_aprovador`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c161_v, COUNT_IF(TRIM(`cod_proximo_aprovador`) RLIKE '^0+([.,]0+)?$') AS c161_z, COUNT_IF(`cod_usuario_grupo_aprovador` IS NULL) AS c162_n, COUNT_IF(`cod_usuario_grupo_aprovador` IS NOT NULL AND LOWER(TRIM(`cod_usuario_grupo_aprovador`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c162_v, COUNT_IF(TRIM(`cod_usuario_grupo_aprovador`) RLIKE '^0+([.,]0+)?$') AS c162_z, COUNT_IF(`ind_delegacao_ativa` IS NULL) AS c163_n, COUNT_IF(`ind_delegacao_ativa` IS NOT NULL AND LOWER(TRIM(`ind_delegacao_ativa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c163_v, COUNT_IF(TRIM(`ind_delegacao_ativa`) RLIKE '^0+([.,]0+)?$') AS c163_z, COUNT_IF(`cod_responsavel_delegacao` IS NULL) AS c164_n, COUNT_IF(`cod_responsavel_delegacao` IS NOT NULL AND LOWER(TRIM(`cod_responsavel_delegacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c164_v, COUNT_IF(TRIM(`cod_responsavel_delegacao`) RLIKE '^0+([.,]0+)?$') AS c164_z, COUNT_IF(`dt_validade_delegacao` IS NULL) AS c165_n, CAST(0 AS BIGINT) AS c165_v, CAST(0 AS BIGINT) AS c165_z, COUNT_IF(`cod_status_atual` IS NULL) AS c166_n, COUNT_IF(`cod_status_atual` IS NOT NULL AND LOWER(TRIM(`cod_status_atual`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c166_v, COUNT_IF(TRIM(`cod_status_atual`) RLIKE '^0+([.,]0+)?$') AS c166_z, COUNT_IF(`cod_status_portal` IS NULL) AS c167_n, COUNT_IF(`cod_status_portal` IS NOT NULL AND LOWER(TRIM(`cod_status_portal`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c167_v, COUNT_IF(TRIM(`cod_status_portal`) RLIKE '^0+([.,]0+)?$') AS c167_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(24, 'cod_tempo_decorrido_01', 'string', c168_n, c168_v, c168_z, 'cod_tempo_decorrido_02', 'string', c169_n, c169_v, c169_z, 'cod_tempo_decorrido_03', 'string', c170_n, c170_v, c170_z, 'cod_tempo_decorrido_04', 'string', c171_n, c171_v, c171_z, 'cod_tempo_decorrido_05', 'string', c172_n, c172_v, c172_z, 'cod_tempo_decorrido_06', 'string', c173_n, c173_v, c173_z, 'cod_tempo_decorrido_07', 'string', c174_n, c174_v, c174_z, 'cod_tempo_decorrido_08', 'string', c175_n, c175_v, c175_z, 'qt_dias_expirar', 'string', c176_n, c176_v, c176_z, 'qt_dias_faltantes_aprovacao', 'string', c177_n, c177_v, c177_z, 'qt_dias_total_aprovacao', 'string', c178_n, c178_v, c178_z, 'ind_eliminacao', 'boolean', c179_n, c179_v, c179_z, 'ind_liberacao', 'string', c180_n, c180_v, c180_z, 'ind_liberacao_incompleta', 'boolean', c181_n, c181_v, c181_z, 'ind_bloqueio_riscos', 'string', c182_n, c182_v, c182_z, 'desc_bloqueio_riscos', 'string', c183_n, c183_v, c183_z, 'vl_indice_definitivo_novo', 'string', c184_n, c184_v, c184_z, 'vl_indice_definitivo_antigo', 'string', c185_n, c185_v, c185_z, 'dt_indice_inflacao', 'date', c186_n, c186_v, c186_z, 'pct_indice_inflacao_novo', 'decimal(17,5)', c187_n, c187_v, c187_z, 'pct_indice_inflacao_antigo', 'decimal(17,5)', c188_n, c188_v, c188_z, 'cod_indice_inflacao', 'string', c189_n, c189_v, c189_z, 'cod_versao_indice_inflacao', 'string', c190_n, c190_v, c190_z, 'system_date', 'string', c191_n, c191_v, c191_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`cod_tempo_decorrido_01` IS NULL) AS c168_n, COUNT_IF(`cod_tempo_decorrido_01` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_01`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c168_v, COUNT_IF(TRIM(`cod_tempo_decorrido_01`) RLIKE '^0+([.,]0+)?$') AS c168_z, COUNT_IF(`cod_tempo_decorrido_02` IS NULL) AS c169_n, COUNT_IF(`cod_tempo_decorrido_02` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_02`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c169_v, COUNT_IF(TRIM(`cod_tempo_decorrido_02`) RLIKE '^0+([.,]0+)?$') AS c169_z, COUNT_IF(`cod_tempo_decorrido_03` IS NULL) AS c170_n, COUNT_IF(`cod_tempo_decorrido_03` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_03`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c170_v, COUNT_IF(TRIM(`cod_tempo_decorrido_03`) RLIKE '^0+([.,]0+)?$') AS c170_z, COUNT_IF(`cod_tempo_decorrido_04` IS NULL) AS c171_n, COUNT_IF(`cod_tempo_decorrido_04` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_04`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c171_v, COUNT_IF(TRIM(`cod_tempo_decorrido_04`) RLIKE '^0+([.,]0+)?$') AS c171_z, COUNT_IF(`cod_tempo_decorrido_05` IS NULL) AS c172_n, COUNT_IF(`cod_tempo_decorrido_05` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_05`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c172_v, COUNT_IF(TRIM(`cod_tempo_decorrido_05`) RLIKE '^0+([.,]0+)?$') AS c172_z, COUNT_IF(`cod_tempo_decorrido_06` IS NULL) AS c173_n, COUNT_IF(`cod_tempo_decorrido_06` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_06`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c173_v, COUNT_IF(TRIM(`cod_tempo_decorrido_06`) RLIKE '^0+([.,]0+)?$') AS c173_z, COUNT_IF(`cod_tempo_decorrido_07` IS NULL) AS c174_n, COUNT_IF(`cod_tempo_decorrido_07` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_07`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c174_v, COUNT_IF(TRIM(`cod_tempo_decorrido_07`) RLIKE '^0+([.,]0+)?$') AS c174_z, COUNT_IF(`cod_tempo_decorrido_08` IS NULL) AS c175_n, COUNT_IF(`cod_tempo_decorrido_08` IS NOT NULL AND LOWER(TRIM(`cod_tempo_decorrido_08`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c175_v, COUNT_IF(TRIM(`cod_tempo_decorrido_08`) RLIKE '^0+([.,]0+)?$') AS c175_z, COUNT_IF(`qt_dias_expirar` IS NULL) AS c176_n, COUNT_IF(`qt_dias_expirar` IS NOT NULL AND LOWER(TRIM(`qt_dias_expirar`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c176_v, COUNT_IF(TRIM(`qt_dias_expirar`) RLIKE '^0+([.,]0+)?$') AS c176_z, COUNT_IF(`qt_dias_faltantes_aprovacao` IS NULL) AS c177_n, COUNT_IF(`qt_dias_faltantes_aprovacao` IS NOT NULL AND LOWER(TRIM(`qt_dias_faltantes_aprovacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c177_v, COUNT_IF(TRIM(`qt_dias_faltantes_aprovacao`) RLIKE '^0+([.,]0+)?$') AS c177_z, COUNT_IF(`qt_dias_total_aprovacao` IS NULL) AS c178_n, COUNT_IF(`qt_dias_total_aprovacao` IS NOT NULL AND LOWER(TRIM(`qt_dias_total_aprovacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c178_v, COUNT_IF(TRIM(`qt_dias_total_aprovacao`) RLIKE '^0+([.,]0+)?$') AS c178_z, COUNT_IF(`ind_eliminacao` IS NULL) AS c179_n, CAST(0 AS BIGINT) AS c179_v, CAST(0 AS BIGINT) AS c179_z, COUNT_IF(`ind_liberacao` IS NULL) AS c180_n, COUNT_IF(`ind_liberacao` IS NOT NULL AND LOWER(TRIM(`ind_liberacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c180_v, COUNT_IF(TRIM(`ind_liberacao`) RLIKE '^0+([.,]0+)?$') AS c180_z, COUNT_IF(`ind_liberacao_incompleta` IS NULL) AS c181_n, CAST(0 AS BIGINT) AS c181_v, CAST(0 AS BIGINT) AS c181_z, COUNT_IF(`ind_bloqueio_riscos` IS NULL) AS c182_n, COUNT_IF(`ind_bloqueio_riscos` IS NOT NULL AND LOWER(TRIM(`ind_bloqueio_riscos`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c182_v, COUNT_IF(TRIM(`ind_bloqueio_riscos`) RLIKE '^0+([.,]0+)?$') AS c182_z, COUNT_IF(`desc_bloqueio_riscos` IS NULL) AS c183_n, COUNT_IF(`desc_bloqueio_riscos` IS NOT NULL AND LOWER(TRIM(`desc_bloqueio_riscos`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c183_v, COUNT_IF(TRIM(`desc_bloqueio_riscos`) RLIKE '^0+([.,]0+)?$') AS c183_z, COUNT_IF(`vl_indice_definitivo_novo` IS NULL) AS c184_n, COUNT_IF(`vl_indice_definitivo_novo` IS NOT NULL AND LOWER(TRIM(`vl_indice_definitivo_novo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c184_v, COUNT_IF(TRIM(`vl_indice_definitivo_novo`) RLIKE '^0+([.,]0+)?$') AS c184_z, COUNT_IF(`vl_indice_definitivo_antigo` IS NULL) AS c185_n, COUNT_IF(`vl_indice_definitivo_antigo` IS NOT NULL AND LOWER(TRIM(`vl_indice_definitivo_antigo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c185_v, COUNT_IF(TRIM(`vl_indice_definitivo_antigo`) RLIKE '^0+([.,]0+)?$') AS c185_z, COUNT_IF(`dt_indice_inflacao` IS NULL) AS c186_n, CAST(0 AS BIGINT) AS c186_v, CAST(0 AS BIGINT) AS c186_z, COUNT_IF(`pct_indice_inflacao_novo` IS NULL) AS c187_n, CAST(0 AS BIGINT) AS c187_v, COUNT_IF(`pct_indice_inflacao_novo` = 0) AS c187_z, COUNT_IF(`pct_indice_inflacao_antigo` IS NULL) AS c188_n, CAST(0 AS BIGINT) AS c188_v, COUNT_IF(`pct_indice_inflacao_antigo` = 0) AS c188_z, COUNT_IF(`cod_indice_inflacao` IS NULL) AS c189_n, COUNT_IF(`cod_indice_inflacao` IS NOT NULL AND LOWER(TRIM(`cod_indice_inflacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c189_v, COUNT_IF(TRIM(`cod_indice_inflacao`) RLIKE '^0+([.,]0+)?$') AS c189_z, COUNT_IF(`cod_versao_indice_inflacao` IS NULL) AS c190_n, COUNT_IF(`cod_versao_indice_inflacao` IS NOT NULL AND LOWER(TRIM(`cod_versao_indice_inflacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c190_v, COUNT_IF(TRIM(`cod_versao_indice_inflacao`) RLIKE '^0+([.,]0+)?$') AS c190_z, COUNT_IF(`system_date` IS NULL) AS c191_n, COUNT_IF(`system_date` IS NOT NULL AND LOWER(TRIM(`system_date`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c191_v, COUNT_IF(TRIM(`system_date`) RLIKE '^0+([.,]0+)?$') AS c191_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a UNION ALL SELECT stack(4, 'timestamp', 'string', c192_n, c192_v, c192_z, 'dateingest', 'date', c193_n, c193_v, c193_z, 'yearingest', 'string', c194_n, c194_v, c194_z, 'monthingest', 'string', c195_n, c195_v, c195_z) AS (coluna, tipo, nulos, vazios, zeros) FROM (SELECT COUNT_IF(`timestamp` IS NULL) AS c192_n, COUNT_IF(`timestamp` IS NOT NULL AND LOWER(TRIM(`timestamp`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c192_v, COUNT_IF(TRIM(`timestamp`) RLIKE '^0+([.,]0+)?$') AS c192_z, COUNT_IF(`dateingest` IS NULL) AS c193_n, CAST(0 AS BIGINT) AS c193_v, CAST(0 AS BIGINT) AS c193_z, COUNT_IF(`yearingest` IS NULL) AS c194_n, COUNT_IF(`yearingest` IS NOT NULL AND LOWER(TRIM(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c194_v, COUNT_IF(TRIM(`yearingest`) RLIKE '^0+([.,]0+)?$') AS c194_z, COUNT_IF(`monthingest` IS NULL) AS c195_n, COUNT_IF(`monthingest` IS NOT NULL AND LOWER(TRIM(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS c195_v, COUNT_IF(TRIM(`monthingest`) RLIKE '^0+([.,]0+)?$') AS c195_z FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a), t AS (SELECT COUNT(*) AS total FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros, t.total - p.nulos - p.vazios AS preenchidos_nao_vazios, CASE WHEN t.total = 0 THEN 'SEM LINHAS' WHEN p.nulos = t.total THEN '100% NULO - CONFERIR COMENTARIO' WHEN p.nulos + p.vazios = t.total THEN '100% SEM VALOR UTIL - CONFERIR' ELSE 'COM DADOS' END AS veredito FROM perf p CROSS JOIN t ORDER BY p.coluna;


## 6.1. Colunas sem fonte

Não há coluna explicitamente classificada como sem fonte no DESCRIBE; não conclui que todas são preenchidas.


In [0]:
SELECT 'Nenhuma coluna explicitamente classificada como sem fonte no DESCRIBE' AS observacao;


## 7. Cardinalidade

Identifica atributos constantes, categorias e possíveis identificadores sem expor valores pessoais.


In [0]:
WITH a AS (SELECT COUNT(*) AS total, COUNT(DISTINCT `cod_fornecedor`) AS d0, COUNT(DISTINCT `num_documento`) AS d1, COUNT(DISTINCT `cod_requisicao_compra`) AS d2, COUNT(DISTINCT `num_material`) AS d3, COUNT(DISTINCT `cod_gl_mm_data_da_aprovacao`) AS d4, COUNT(DISTINCT `tp_tipo_documento`) AS d5, COUNT(DISTINCT `cod_evento`) AS d6, COUNT(DISTINCT `cod_empresa`) AS d7, COUNT(DISTINCT `cod_exercicio`) AS d8, COUNT(DISTINCT `cod_unidade_negocios`) AS d9, COUNT(DISTINCT `cod_centro`) AS d10 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) SELECT coluna, distintos, total, CASE WHEN distintos <= 1 THEN 'CONSTANTE OU VAZIO' WHEN distintos <= 20 THEN 'BAIXA CARDINALIDADE' WHEN distintos >= total * 0.9 THEN 'CANDIDATO A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS perfil FROM (SELECT total, stack(11, 'cod_fornecedor', d0, 'num_documento', d1, 'cod_requisicao_compra', d2, 'num_material', d3, 'cod_gl_mm_data_da_aprovacao', d4, 'tp_tipo_documento', d5, 'cod_evento', d6, 'cod_empresa', d7, 'cod_exercicio', d8, 'cod_unidade_negocios', d9, 'cod_centro', d10) AS (coluna, distintos) FROM a) s ORDER BY distintos DESC;


## 8. Domínio

Mostra até oito valores por campo categórico não pessoal; nomes, identificadores pessoais e endereços ficam fora.


In [0]:
WITH d AS (SELECT 'cod_gl_mm_data_da_aprovacao' AS coluna, CAST(`cod_gl_mm_data_da_aprovacao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_gl_mm_data_da_aprovacao` UNION ALL SELECT 'tp_tipo_documento' AS coluna, CAST(`tp_tipo_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `tp_tipo_documento` UNION ALL SELECT 'cod_evento' AS coluna, CAST(`cod_evento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_evento` UNION ALL SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_empresa` UNION ALL SELECT 'cod_exercicio' AS coluna, CAST(`cod_exercicio` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_exercicio` UNION ALL SELECT 'cod_unidade_negocios' AS coluna, CAST(`cod_unidade_negocios` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_unidade_negocios` UNION ALL SELECT 'cod_requisicao_compra' AS coluna, CAST(`cod_requisicao_compra` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_requisicao_compra` UNION ALL SELECT 'cod_item_requisicao_compra' AS coluna, CAST(`cod_item_requisicao_compra` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_item_requisicao_compra`), r AS (SELECT coluna, valor, linhas, ROW_NUMBER() OVER (PARTITION BY coluna ORDER BY linhas DESC, valor) AS pos FROM d) SELECT coluna, valor, linhas FROM r WHERE pos <= 8 ORDER BY coluna, pos;


## 9. Booleanas e indicadores

Mede os estados observados, sem converter indicadores STRING em booleanos por suposição.


In [0]:
WITH a AS (SELECT 'tp_tipo_documento' AS coluna, CAST(`tp_tipo_documento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `tp_tipo_documento` UNION ALL SELECT 'tp_tipo_pedido' AS coluna, CAST(`tp_tipo_pedido` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `tp_tipo_pedido` UNION ALL SELECT 'cod_status_liberacao' AS coluna, CAST(`cod_status_liberacao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_status_liberacao` UNION ALL SELECT 'tp_movimento' AS coluna, CAST(`tp_movimento` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `tp_movimento` UNION ALL SELECT 'ind_delegacao_ativa' AS coluna, CAST(`ind_delegacao_ativa` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `ind_delegacao_ativa` UNION ALL SELECT 'cod_status_atual' AS coluna, CAST(`cod_status_atual` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_status_atual` UNION ALL SELECT 'cod_status_portal' AS coluna, CAST(`cod_status_portal` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_status_portal` UNION ALL SELECT 'ind_eliminacao' AS coluna, CAST(`ind_eliminacao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `ind_eliminacao` UNION ALL SELECT 'ind_liberacao' AS coluna, CAST(`ind_liberacao` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `ind_liberacao` UNION ALL SELECT 'ind_liberacao_incompleta' AS coluna, CAST(`ind_liberacao_incompleta` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `ind_liberacao_incompleta` UNION ALL SELECT 'ind_bloqueio_riscos' AS coluna, CAST(`ind_bloqueio_riscos` AS STRING) AS valor, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `ind_bloqueio_riscos`) SELECT coluna, valor, linhas FROM a ORDER BY coluna, linhas DESC;


## 10. Perfil numérico

Perfila quantidades e valores. Campos de códigos numéricos não são tratados como montantes.


In [0]:
SELECT * FROM (SELECT stack(8, 'vl_valor_bruto', CAST(n0_min AS DOUBLE), CAST(n0_max AS DOUBLE), CAST(n0_avg AS DOUBLE), CAST(n0_med AS DOUBLE), n0_neg, n0_zero, 'vl_limite_alcada', CAST(n1_min AS DOUBLE), CAST(n1_max AS DOUBLE), CAST(n1_avg AS DOUBLE), CAST(n1_med AS DOUBLE), n1_neg, n1_zero, 'qt_quantidade', CAST(n2_min AS DOUBLE), CAST(n2_max AS DOUBLE), CAST(n2_avg AS DOUBLE), CAST(n2_med AS DOUBLE), n2_neg, n2_zero, 'vl_fixado_cabecalho', CAST(n3_min AS DOUBLE), CAST(n3_max AS DOUBLE), CAST(n3_avg AS DOUBLE), CAST(n3_med AS DOUBLE), n3_neg, n3_zero, 'vl_preco_liquido_ped', CAST(n4_min AS DOUBLE), CAST(n4_max AS DOUBLE), CAST(n4_avg AS DOUBLE), CAST(n4_med AS DOUBLE), n4_neg, n4_zero, 'vl_preco_bruto', CAST(n5_min AS DOUBLE), CAST(n5_max AS DOUBLE), CAST(n5_avg AS DOUBLE), CAST(n5_med AS DOUBLE), n5_neg, n5_zero, 'pct_indice_inflacao_novo', CAST(n6_min AS DOUBLE), CAST(n6_max AS DOUBLE), CAST(n6_avg AS DOUBLE), CAST(n6_med AS DOUBLE), n6_neg, n6_zero, 'pct_indice_inflacao_antigo', CAST(n7_min AS DOUBLE), CAST(n7_max AS DOUBLE), CAST(n7_avg AS DOUBLE), CAST(n7_med AS DOUBLE), n7_neg, n7_zero) AS (coluna, minimo, maximo, media, mediana_aproximada, negativos, zeros) FROM (SELECT MIN(`vl_valor_bruto`) AS n0_min, MAX(`vl_valor_bruto`) AS n0_max, AVG(`vl_valor_bruto`) AS n0_avg, PERCENTILE_APPROX(`vl_valor_bruto`, 0.5) AS n0_med, COUNT_IF(`vl_valor_bruto` < 0) AS n0_neg, COUNT_IF(`vl_valor_bruto` = 0) AS n0_zero, MIN(`vl_limite_alcada`) AS n1_min, MAX(`vl_limite_alcada`) AS n1_max, AVG(`vl_limite_alcada`) AS n1_avg, PERCENTILE_APPROX(`vl_limite_alcada`, 0.5) AS n1_med, COUNT_IF(`vl_limite_alcada` < 0) AS n1_neg, COUNT_IF(`vl_limite_alcada` = 0) AS n1_zero, MIN(`qt_quantidade`) AS n2_min, MAX(`qt_quantidade`) AS n2_max, AVG(`qt_quantidade`) AS n2_avg, PERCENTILE_APPROX(`qt_quantidade`, 0.5) AS n2_med, COUNT_IF(`qt_quantidade` < 0) AS n2_neg, COUNT_IF(`qt_quantidade` = 0) AS n2_zero, MIN(`vl_fixado_cabecalho`) AS n3_min, MAX(`vl_fixado_cabecalho`) AS n3_max, AVG(`vl_fixado_cabecalho`) AS n3_avg, PERCENTILE_APPROX(`vl_fixado_cabecalho`, 0.5) AS n3_med, COUNT_IF(`vl_fixado_cabecalho` < 0) AS n3_neg, COUNT_IF(`vl_fixado_cabecalho` = 0) AS n3_zero, MIN(`vl_preco_liquido_ped`) AS n4_min, MAX(`vl_preco_liquido_ped`) AS n4_max, AVG(`vl_preco_liquido_ped`) AS n4_avg, PERCENTILE_APPROX(`vl_preco_liquido_ped`, 0.5) AS n4_med, COUNT_IF(`vl_preco_liquido_ped` < 0) AS n4_neg, COUNT_IF(`vl_preco_liquido_ped` = 0) AS n4_zero, MIN(`vl_preco_bruto`) AS n5_min, MAX(`vl_preco_bruto`) AS n5_max, AVG(`vl_preco_bruto`) AS n5_avg, PERCENTILE_APPROX(`vl_preco_bruto`, 0.5) AS n5_med, COUNT_IF(`vl_preco_bruto` < 0) AS n5_neg, COUNT_IF(`vl_preco_bruto` = 0) AS n5_zero, MIN(`pct_indice_inflacao_novo`) AS n6_min, MAX(`pct_indice_inflacao_novo`) AS n6_max, AVG(`pct_indice_inflacao_novo`) AS n6_avg, PERCENTILE_APPROX(`pct_indice_inflacao_novo`, 0.5) AS n6_med, COUNT_IF(`pct_indice_inflacao_novo` < 0) AS n6_neg, COUNT_IF(`pct_indice_inflacao_novo` = 0) AS n6_zero, MIN(`pct_indice_inflacao_antigo`) AS n7_min, MAX(`pct_indice_inflacao_antigo`) AS n7_max, AVG(`pct_indice_inflacao_antigo`) AS n7_avg, PERCENTILE_APPROX(`pct_indice_inflacao_antigo`, 0.5) AS n7_med, COUNT_IF(`pct_indice_inflacao_antigo` < 0) AS n7_neg, COUNT_IF(`pct_indice_inflacao_antigo` = 0) AS n7_zero FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a) n ORDER BY coluna;


## 11. Totais para conciliação

Soma separadamente cada quantidade e valor. Totais DOUBLE são aproximados; comparar moedas e granularidade antes de conciliar.


In [0]:
SELECT coluna, total FROM (SELECT stack(8, 'vl_valor_bruto', CAST(s0 AS DOUBLE), 'vl_limite_alcada', CAST(s1 AS DOUBLE), 'qt_quantidade', CAST(s2 AS DOUBLE), 'vl_fixado_cabecalho', CAST(s3 AS DOUBLE), 'vl_preco_liquido_ped', CAST(s4 AS DOUBLE), 'vl_preco_bruto', CAST(s5 AS DOUBLE), 'pct_indice_inflacao_novo', CAST(s6 AS DOUBLE), 'pct_indice_inflacao_antigo', CAST(s7 AS DOUBLE)) AS (coluna, total) FROM (SELECT SUM(`vl_valor_bruto`) AS s0, SUM(`vl_limite_alcada`) AS s1, SUM(`qt_quantidade`) AS s2, SUM(`vl_fixado_cabecalho`) AS s3, SUM(`vl_preco_liquido_ped`) AS s4, SUM(`vl_preco_bruto`) AS s5, SUM(`pct_indice_inflacao_novo`) AS s6, SUM(`pct_indice_inflacao_antigo`) AS s7 FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) a) s ORDER BY coluna;


## 12. Datas

Verifica sentinelas e formatos; mínimos de campos STRING são literais, não datas cronológicas.


In [0]:
SELECT * FROM (SELECT 'dt_recebimento_01' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_01` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_01` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_01`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_01`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_02' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_02` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_02` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_02`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_02`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_03' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_03` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_03` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_03`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_03`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_04' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_04` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_04` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_04`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_04`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_05' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_05` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_05` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_05`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_05`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_06' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_06` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_06` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_06`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_06`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_07' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_07` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_07` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_07`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_07`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_08' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_08` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_08` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_08`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_08`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_recebimento_msg' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_recebimento_msg` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_recebimento_msg` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_recebimento_msg`) AS STRING) AS menor_literal, CAST(MAX(`dt_recebimento_msg`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_01' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_01` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_01` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_01`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_01`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_02' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_02` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_02` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_02`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_02`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_03' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_03` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_03` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_03`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_03`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_04' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_04` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_04` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_04`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_04`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_05' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_05` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_05` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_05`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_05`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_06' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_06` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_06` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_06`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_06`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_07' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_07` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_07` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_07`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_07`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_08' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_08` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_08` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_08`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_08`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_liberacao_final' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_final` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_liberacao_final` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_liberacao_final`) AS STRING) AS menor_literal, CAST(MAX(`dt_liberacao_final`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_lancamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_lancamento` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_lancamento` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_lancamento`) AS STRING) AS menor_literal, CAST(MAX(`dt_lancamento`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_remessa' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_remessa` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_remessa` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_remessa`) AS STRING) AS menor_literal, CAST(MAX(`dt_remessa`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_validade_delegacao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_validade_delegacao` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_validade_delegacao` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_validade_delegacao`) AS STRING) AS menor_literal, CAST(MAX(`dt_validade_delegacao`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dt_indice_inflacao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_indice_inflacao` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dt_indice_inflacao` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dt_indice_inflacao`) AS STRING) AS menor_literal, CAST(MAX(`dt_indice_inflacao`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'dateingest' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dateingest` IS NULL) AS nulos, CAST(0 AS BIGINT) AS vazios, CAST(0 AS BIGINT) AS sentinela_00000000, COUNT_IF(CAST(`dateingest` AS DATE) = DATE '9999-12-31') AS sentinela_9999_12_31, CAST(0 AS BIGINT) AS formato_dd_mm_aaaa, CAST(0 AS BIGINT) AS formato_aaaammdd, CAST(MIN(`dateingest`) AS STRING) AS menor_literal, CAST(MAX(`dateingest`) AS STRING) AS maior_literal FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) d ORDER BY coluna;


## 13. Códigos

Detecta zeros à esquerda e possíveis colisões ao normalizar identificadores não pessoais. Não normalizar sem avaliar a semântica.


In [0]:
SELECT *, codigos_numericos_distintos - codigos_normalizados_distintos AS colisoes_potenciais FROM (SELECT 'cod_gl_mm_data_da_aprovacao' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_gl_mm_data_da_aprovacao`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_gl_mm_data_da_aprovacao`))) AS tamanho_max, COUNT_IF(TRIM(`cod_gl_mm_data_da_aprovacao`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_gl_mm_data_da_aprovacao`) RLIKE '^[0-9]+$' THEN TRIM(`cod_gl_mm_data_da_aprovacao`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_gl_mm_data_da_aprovacao`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_gl_mm_data_da_aprovacao`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'num_documento' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_documento`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_documento`))) AS tamanho_max, COUNT_IF(TRIM(`num_documento`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_documento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'cod_empresa' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_empresa`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_empresa`))) AS tamanho_max, COUNT_IF(TRIM(`cod_empresa`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN TRIM(`cod_empresa`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_empresa`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_empresa`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'cod_exercicio' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_exercicio`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_exercicio`))) AS tamanho_max, COUNT_IF(TRIM(`cod_exercicio`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_exercicio`) RLIKE '^[0-9]+$' THEN TRIM(`cod_exercicio`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_exercicio`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_exercicio`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'cod_unidade_negocios' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_unidade_negocios`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_unidade_negocios`))) AS tamanho_max, COUNT_IF(TRIM(`cod_unidade_negocios`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_negocios`) RLIKE '^[0-9]+$' THEN TRIM(`cod_unidade_negocios`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_unidade_negocios`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_unidade_negocios`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'num_documento_inventario' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_documento_inventario`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_documento_inventario`))) AS tamanho_max, COUNT_IF(TRIM(`num_documento_inventario`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_documento_inventario`) RLIKE '^[0-9]+$' THEN TRIM(`num_documento_inventario`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_documento_inventario`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_documento_inventario`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'cod_requisicao_compra' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_requisicao_compra`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_requisicao_compra`))) AS tamanho_max, COUNT_IF(TRIM(`cod_requisicao_compra`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_requisicao_compra`) RLIKE '^[0-9]+$' THEN TRIM(`cod_requisicao_compra`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_requisicao_compra`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_requisicao_compra`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'cod_item_requisicao_compra' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_item_requisicao_compra`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_item_requisicao_compra`))) AS tamanho_max, COUNT_IF(TRIM(`cod_item_requisicao_compra`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_item_requisicao_compra`) RLIKE '^[0-9]+$' THEN TRIM(`cod_item_requisicao_compra`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_item_requisicao_compra`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_item_requisicao_compra`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'num_acompanhamento' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`num_acompanhamento`))) AS tamanho_min, MAX(LENGTH(TRIM(`num_acompanhamento`))) AS tamanho_max, COUNT_IF(TRIM(`num_acompanhamento`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`num_acompanhamento`) RLIKE '^[0-9]+$' THEN TRIM(`num_acompanhamento`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`num_acompanhamento`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`num_acompanhamento`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` UNION ALL SELECT 'cod_classificacao_contabil_req' AS coluna, COUNT(*) AS linhas, MIN(LENGTH(TRIM(`cod_classificacao_contabil_req`))) AS tamanho_min, MAX(LENGTH(TRIM(`cod_classificacao_contabil_req`))) AS tamanho_max, COUNT_IF(TRIM(`cod_classificacao_contabil_req`) RLIKE '^0+[0-9]+$') AS com_zeros_a_esquerda, COUNT(DISTINCT CASE WHEN TRIM(`cod_classificacao_contabil_req`) RLIKE '^[0-9]+$' THEN TRIM(`cod_classificacao_contabil_req`) END) AS codigos_numericos_distintos, COUNT(DISTINCT CASE WHEN TRIM(`cod_classificacao_contabil_req`) RLIKE '^[0-9]+$' THEN REGEXP_REPLACE(TRIM(`cod_classificacao_contabil_req`), '^0+(?=[0-9])', '') END) AS codigos_normalizados_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) c ORDER BY coluna;


## 14. Checksum

Conta linhas 100% idênticas por hash da linha inteira; não retorna conteúdo nem hashes de dados pessoais.


In [0]:
WITH h AS (SELECT md5(to_json(struct(t.`cod_gl_mm_data_da_aprovacao`, t.`num_documento`, t.`tp_tipo_documento`, t.`num_sequencial_alteracoes`, t.`cod_evento`, t.`cod_empresa`, t.`nm_empresa`, t.`cod_exercicio`, t.`cod_unidade_negocios`, t.`num_documento_inventario`, t.`cod_requisicao_compra`, t.`cod_item_requisicao_compra`, t.`num_acompanhamento`, t.`cod_classificacao_contabil_req`, t.`cod_classe_custo_req`, t.`num_ordem_req`, t.`desc_texto_breve_req`, t.`num_pedido_compras`, t.`num_item_pedido_compras`, t.`tp_tipo_pedido`, t.`cod_classificacao_contabil`, t.`cod_classe_custo`, t.`cod_classificacao_contabil_ped`, t.`cod_classe_custo_ped`, t.`num_ordem_ped`, t.`desc_texto_breve_ped`, t.`num_ordem`, t.`num_ordem_interna`, t.`cod_categoria_item`, t.`cod_elemento_pep`, t.`num_contrato`, t.`num_item_contrato`, t.`num_contrato_re`, t.`num_item_contrato_re`, t.`cod_fornecedor`, t.`cod_fornecedor_re`, t.`nm_fornecedor_re`, t.`num_material`, t.`desc_texto_breve`, t.`num_folha_servicos`, t.`num_servico`, t.`num_servico_re`, t.`sg_unidade_medida`, t.`sg_unidade_medida_base`, t.`cod_centro`, t.`cod_centro_ped`, t.`cod_centro_custo`, t.`desc_denominacao`, t.`desc_texto_breve_re`, t.`num_local_instalacao`, t.`num_equipamento`, t.`cod_org_compras`, t.`cod_grupo_compradores`, t.`vl_valor_bruto`, t.`vl_limite_alcada`, t.`qt_quantidade`, t.`vl_fixado_cabecalho`, t.`vl_preco_liquido_ped`, t.`vl_preco_bruto`, t.`sg_moeda`, t.`sg_moeda_ped`, t.`cod_estrategia_liberacao`, t.`cod_status_liberacao`, t.`cod_grupo_liberacao`, t.`cod_grupo_liberador`, t.`cod_liberacao`, t.`cod_nivel_alcada`, t.`cod_meio_aprovacao`, t.`tp_movimento`, t.`desc_cod_liberacao_01`, t.`desc_cod_liberacao_02`, t.`desc_cod_liberacao_03`, t.`desc_cod_liberacao_04`, t.`desc_cod_liberacao_05`, t.`desc_cod_liberacao_06`, t.`desc_cod_liberacao_07`, t.`desc_cod_liberacao_08`, t.`cod_cadeia_aprovacao_n1`, t.`cod_cadeia_aprovacao_n2`, t.`cod_cadeia_aprovacao_n3`, t.`cod_cadeia_aprovacao_n4`, t.`cod_cadeia_aprovacao_n5`, t.`cod_cadeia_aprovacao_n6`, t.`cod_cadeia_aprovacao_n7`, t.`cod_cadeia_aprovacao_n8`, t.`cod_nivel_n1`, t.`cod_nivel_n2`, t.`cod_nivel_n3`, t.`cod_nivel_n4`, t.`cod_nivel_n5`, t.`cod_nivel_n6`, t.`cod_nivel_n7`, t.`cod_nivel_n8`, t.`nm_aprovador_01`, t.`nm_aprovador_02`, t.`nm_aprovador_03`, t.`nm_aprovador_04`, t.`nm_aprovador_05`, t.`nm_aprovador_06`, t.`nm_aprovador_07`, t.`nm_aprovador_08`, t.`cod_liberacao_01`, t.`cod_liberacao_02`, t.`cod_liberacao_03`, t.`cod_liberacao_04`, t.`cod_liberacao_05`, t.`cod_liberacao_06`, t.`cod_liberacao_07`, t.`cod_liberacao_08`, t.`cod_usuario_lib_01`, t.`cod_usuario_lib_02`, t.`cod_usuario_lib_03`, t.`cod_usuario_lib_04`, t.`cod_usuario_lib_05`, t.`cod_usuario_lib_06`, t.`cod_usuario_lib_07`, t.`cod_usuario_lib_08`, t.`dt_recebimento_01`, t.`dt_recebimento_02`, t.`dt_recebimento_03`, t.`dt_recebimento_04`, t.`dt_recebimento_05`, t.`dt_recebimento_06`, t.`dt_recebimento_07`, t.`dt_recebimento_08`, t.`hr_recebimento_01`, t.`hr_recebimento_02`, t.`hr_recebimento_03`, t.`hr_recebimento_04`, t.`hr_recebimento_05`, t.`hr_recebimento_06`, t.`hr_recebimento_07`, t.`hr_recebimento_08`, t.`dt_recebimento_msg`, t.`hr_recebimento_msg`, t.`dt_liberacao_01`, t.`dt_liberacao_02`, t.`dt_liberacao_03`, t.`dt_liberacao_04`, t.`dt_liberacao_05`, t.`dt_liberacao_06`, t.`dt_liberacao_07`, t.`dt_liberacao_08`, t.`dt_liberacao_final`, t.`hr_liberacao_01`, t.`hr_liberacao_02`, t.`hr_liberacao_03`, t.`hr_liberacao_04`, t.`hr_liberacao_05`, t.`hr_liberacao_06`, t.`hr_liberacao_07`, t.`hr_liberacao_08`, t.`hr_liberacao_final`, t.`hr_entrada`, t.`dt_lancamento`, t.`dt_remessa`, t.`nm_requisitante`, t.`nm_solicitante`, t.`nm_usuario_aprovador`, t.`cod_criado_por`, t.`cod_usuario_substituido`, t.`cod_proximo_aprovador`, t.`cod_usuario_grupo_aprovador`, t.`ind_delegacao_ativa`, t.`cod_responsavel_delegacao`, t.`dt_validade_delegacao`, t.`cod_status_atual`, t.`cod_status_portal`, t.`cod_tempo_decorrido_01`, t.`cod_tempo_decorrido_02`, t.`cod_tempo_decorrido_03`, t.`cod_tempo_decorrido_04`, t.`cod_tempo_decorrido_05`, t.`cod_tempo_decorrido_06`, t.`cod_tempo_decorrido_07`, t.`cod_tempo_decorrido_08`, t.`qt_dias_expirar`, t.`qt_dias_faltantes_aprovacao`, t.`qt_dias_total_aprovacao`, t.`ind_eliminacao`, t.`ind_liberacao`, t.`ind_liberacao_incompleta`, t.`ind_bloqueio_riscos`, t.`desc_bloqueio_riscos`, t.`vl_indice_definitivo_novo`, t.`vl_indice_definitivo_antigo`, t.`dt_indice_inflacao`, t.`pct_indice_inflacao_novo`, t.`pct_indice_inflacao_antigo`, t.`cod_indice_inflacao`, t.`cod_versao_indice_inflacao`, t.`system_date`, t.`timestamp`, t.`dateingest`, t.`yearingest`, t.`monthingest`), map('ignoreNullFields', 'false'))) AS hash_linha FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` t), a AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha) SELECT COALESCE(SUM(linhas),0) AS linhas, COUNT(*) AS linhas_distintas, COALESCE(SUM(linhas - 1),0) AS repeticoes_identicas FROM a;


## 15. Amostra

Mostra 20 registros com colunas não pessoais; campos potencialmente pessoais são omitidos ou hash de 8 caracteres.


In [0]:
SELECT `cod_gl_mm_data_da_aprovacao`, `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`, `cod_evento`, `cod_empresa`, `nm_empresa`, `cod_exercicio`, `cod_unidade_negocios`, `num_documento_inventario`, `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_acompanhamento`, `cod_classificacao_contabil_req`, `cod_classe_custo_req`, `num_ordem_req`, SUBSTRING(MD5(CAST(`nm_fornecedor_re` AS STRING)),1,8) AS `nm_fornecedor_re_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_01` AS STRING)),1,8) AS `nm_aprovador_01_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_02` AS STRING)),1,8) AS `nm_aprovador_02_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_03` AS STRING)),1,8) AS `nm_aprovador_03_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_04` AS STRING)),1,8) AS `nm_aprovador_04_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` LIMIT 20;


## 16. Distribuição interna

Compara o mix de categorias e centro, quando houver, sem expor pessoas.


In [0]:
SELECT `cod_centro`, `cod_gl_mm_data_da_aprovacao`, `tp_tipo_documento`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `cod_centro`, `cod_gl_mm_data_da_aprovacao`, `tp_tipo_documento` ORDER BY linhas DESC LIMIT 100;


## 17. Freshness

Mede datas de carga e partições; não confundir carimbo de ingestão com evento de negócio.


In [0]:
SELECT COUNT(*) AS linhas, MIN(`dateingest`) AS `min_dateingest`, MAX(`dateingest`) AS `max_dateingest`, COUNT(DISTINCT `dateingest`) AS `distintos_dateingest`, MIN(`yearingest`) AS `min_yearingest`, MAX(`yearingest`) AS `max_yearingest`, COUNT(DISTINCT `yearingest`) AS `distintos_yearingest`, MIN(`monthingest`) AS `min_monthingest`, MAX(`monthingest`) AS `max_monthingest`, COUNT(DISTINCT `monthingest`) AS `distintos_monthingest` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`;


## 18. Análises específicas da transação

Testes derivados dos comentários do DESCRIBE; as hipóteses dependem dos resultados medidos.


### 18.1. Etapas de aprovação e integridade temporal


In [0]:
SELECT `tp_tipo_documento`, `cod_status_atual`, COUNT(*) AS linhas, COUNT_IF(`dt_liberacao_final` IS NOT NULL) AS liberacao_final_preenchida, COUNT_IF(`dt_recebimento_01` IS NOT NULL AND `dt_liberacao_01` IS NOT NULL AND `dt_liberacao_01` < `dt_recebimento_01`) AS liberacao_n1_antes_recebimento, COUNT_IF(`dt_liberacao_07` IS NOT NULL OR `dt_liberacao_08` IS NOT NULL) AS eventos_niveis_7_8, COUNT_IF(`ind_liberacao_incompleta` = TRUE) AS liberacao_incompleta FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `tp_tipo_documento`, `cod_status_atual` ORDER BY linhas DESC LIMIT 100;


## 19. Amostra ampliada

Amostra de até 500 registros sem campos pessoais literais; não equivale à extração completa do cenário.


In [0]:
SELECT `cod_gl_mm_data_da_aprovacao`, `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`, `cod_evento`, `cod_empresa`, `nm_empresa`, `cod_exercicio`, `cod_unidade_negocios`, `num_documento_inventario`, `cod_requisicao_compra`, `cod_item_requisicao_compra`, `num_acompanhamento`, `cod_classificacao_contabil_req`, `cod_classe_custo_req`, `num_ordem_req`, SUBSTRING(MD5(CAST(`nm_fornecedor_re` AS STRING)),1,8) AS `nm_fornecedor_re_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_01` AS STRING)),1,8) AS `nm_aprovador_01_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_02` AS STRING)),1,8) AS `nm_aprovador_02_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_03` AS STRING)),1,8) AS `nm_aprovador_03_hash`, SUBSTRING(MD5(CAST(`nm_aprovador_04` AS STRING)),1,8) AS `nm_aprovador_04_hash` FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` LIMIT 500;


## 20. Resumo

Copie este bloco junto com as seções 3, 5, 6 e 18 para validar chave real e filtros antes dos cenários.


In [0]:
WITH d AS (SELECT `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`, COUNT(*) AS linhas FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao` GROUP BY `num_documento`, `tp_tipo_documento`, `num_sequencial_alteracoes`), v AS (SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_centro`) AS centros_distintos FROM `dev_procurement`.`corp_curated`.`tbl_ds_pro_zvcmm103_parada_manutencao`) SELECT 'ZVCMM103' AS transacao, 'tbl_ds_pro_zvcmm103_parada_manutencao' AS objeto, v.linhas, (SELECT COUNT(*) FROM d) AS chaves_candidatas_distintas, (SELECT COUNT(*) FROM d WHERE linhas > 1) AS chaves_candidatas_repetidas, v.centros_distintos, 'Chave candidata: num_documento + tp_tipo_documento + num_sequencial_alteracoes' AS nota_chave FROM v;


## Checklist para comparação com SAP

- [ ] Confirmar comentário do objeto e filtros aplicados na tela SAP.
- [ ] Registrar instante de execução e mesmo recorte temporal da extração SAP.
- [ ] Validar chave real, duplicatas idênticas e versões distintas.
- [ ] Conferir centros 4128 e 4014 somente quando há campo de centro; caso contrário, definir correspondência de negócio antes de comparar.
- [ ] Verificar nulos sem declaração de sem fonte; não classificar lacuna documentada como defeito.
- [ ] Normalizar códigos com zeros à esquerda e datas; validar separador decimal, moeda e tolerância de 0,005 para DOUBLE.
- [ ] Não extrair nem divulgar valores pessoais literais.
- [ ] Executar cenários somente após receber os resultados deste notebook.
